# 11 — Thử nghiệm mở rộng kiến trúc: state-space, Mamba, OMI-CNN (STEMI)

**Dự án ACS-ECG-AI (Vinmec)** · dùng lại nguyên vẹn pipeline dữ liệu/K-Fold/hiệu chuẩn/TEST của
[`08_stemi_stability_calibration.ipynb`](08_stemi_stability_calibration.ipynb), chỉ mở rộng danh
sách kiến trúc để trả lời câu hỏi: **5 kiến trúc mới (chưa từng chạy ở notebook 08) có đáng dùng
để phát hiện STEMI không?** Notebook này **CHỈ huấn luyện 5 model mới** — không lặp lại 8 model
đã có kết quả K-Fold ở notebook 08 (`PlainCNN`, `XResNet1D`, `ConvNeXtV2_1D`, `AiTiAMI`, `ResNet1D`,
`SEResNet1D`, `CNN+BiLSTM`, `InceptionTime1D`), để tránh tốn GPU train lại thứ đã có sẵn.

## Phạm vi 5 model mới

| Model | Bản chất | Giới hạn cần biết |
|---|---|---|
| `S4D1D` | State-space model (Gu et al. 2022, S4D — bản diagonal) | Cài lại **thuần PyTorch bằng FFT**, không dùng gói `s4`/kernel CUDA gốc (gói gốc hay lỗi khi build trên Colab). Đúng công thức toán, nhưng tốc độ không đại diện đầy đủ cho bản CUDA chính thức. |
| `Mamba2Lite1D` | Selective state-space (Gu & Dao 2023, nền tảng của Mamba/Mamba-2) | Cài lại **thuần PyTorch, scan tuần tự bằng vòng lặp Python** — không dùng gói `mamba-ssm`/`causal-conv1d` (cần biên dịch CUDA riêng). Không phải bản Mamba-2 chính thức, chỉ xấp xỉ đúng công thức. |
| `ECGMamba1D` | CNN + Mamba-lite (lấy cảm hứng từ Liu et al. 2024, ECGMamba) | Không có mã nguồn gốc công khai để xác minh — tự thiết kế lại theo đúng mô tả kiến trúc 2 tầng (CNN rút đặc trưng cục bộ rồi Mamba mô hình hoá quan hệ dài hạn), không phải bản sao. |
| `OMICNN1D` | CNN multi-scale (lấy cảm hứng từ Queen of Hearts, Powerful Medical/PMcardio) | Model thương mại, **không công bố mã nguồn/kiến trúc** — đây là thiết kế tự làm theo mô tả công khai (CNN đa nhánh kernel song song), không phải bản sao trọng số/kiến trúc thật. |
| `OSCNN1D` | Omni-Scale CNN (Tang, Long, Liu, Yin 2022, *Omni-Scale CNNs*) | Mỗi tầng chạy song song một nhánh conv cho **mỗi kernel size nguyên tố** trong một dải cho trước, phủ hết các scale thay vì dò kernel size thủ công. Cài lại theo đúng thuật toán gốc (sinh kernel + ngân sách tham số tự co), không dùng lại mã nguồn tác giả. |

**Không đưa vào notebook này** (khác nhóm S4D/Mamba ở trên vì đây là các *pretrained foundation
model*, không phải kiến trúc huấn luyện từ đầu): `ECGFounder`, `HuBERT-ECG`, `ST-MEM`, `ECG-JEPA`,
`ECG-CPC`, `xECG`. Lý do: mỗi model này cần một checkpoint đã pretrain (không có sẵn trong dự án,
chưa được tải và xác minh license/tính đúng đắn ở đây) và thường yêu cầu tiền xử lý input khác với
z-score 12×5000 hiện tại (ví dụ `ECGFounder` dùng chuẩn hoá riêng của nhóm tác giả). Muốn thêm các
model này, cần plug-in checkpoint đã tải + hàm tiền xử lý riêng — không thể huấn luyện lại từ đầu
như các kiến trúc trong notebook này (sẽ không tái tạo được năng lực đã pretrain).

## Giữ nguyên từ notebook 08

Toàn bộ pipeline dữ liệu, chia POOL/TEST (85/15), 5-fold theo bệnh nhân, kiểm định ghép cặp, ép
ngưỡng Sensitivity, hiệu chuẩn cross-fit, phân tầng nguy cơ, đánh giá TEST giữ riêng (bagging và
model FINAL) — **không đổi gì** so với notebook 08, chỉ đổi `CANDIDATES` (5 model MỚI, thay hoàn
toàn 8 model cũ chứ không cộng dồn) và đường dẫn checkpoint/OOF riêng để không ghi đè kết quả của
notebook 08. Mục 17(a) vẫn nạp lại checkpoint cũ của notebook 03 (10 model, gồm cả 8 model của
notebook 08 + `TCN1D`/`Transformer1D`) để đối chiếu tham khảo trên val 85/15 gốc — xem ghi chú ở
mục đó về việc hai nguồn dữ liệu không so trực tiếp được.

**Lưu ý chi phí:** 5 kiến trúc × 5 fold × 30 epoch — ít hơn notebook 08 (8 kiến trúc) về số model
nhưng chậm hơn trên mỗi model vì `Mamba2Lite1D`/`ECGMamba1D` đặc biệt chậm (scan tuần tự theo thời
gian bằng vòng lặp Python, không tận dụng được tối ưu CUDA như bản `mamba-ssm` gốc).

**Tập test ẩn 10% (của toàn bộ dự án, không phải TEST 15% giữ riêng ở mục 12b) không được chạm
tới ở bất kỳ bước nào.**

## 1. Cài thư viện

Chỉ cài `wfdb` nếu môi trường chưa có. Trên Colab cần cài lại mỗi phiên (~10 giây).

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("wfdb") is None:
    print("Đang cài wfdb ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wfdb"], check=True)

import wfdb

print("wfdb", wfdb.__version__)

## 2. Cấu hình

Notebook chạy được trên **Google Colab** và **máy cá nhân**, tự nhận diện môi trường.
Chỉ cần sửa đúng một chỗ:

- **Colab** → `DRIVE_PROJECT` (tên thư mục trên Drive chứa `datasets.zip`)
- **Máy cá nhân** → `LOCAL_DATA_ROOT`

Ngoài ra đổi `RUN_MODE` khi muốn chuyển từ chạy thử sang chạy thật.

**Thư mục `outputs/` chỉ giữ hai thứ:** checkpoint mô hình và cache tín hiệu. Hình và bảng số
đều hiển thị ngay trong notebook nên không ghi ra file nữa.

Trên Colab, cache đặt ở `/content` (nhanh, mất khi hết phiên) và được sao lưu lên Drive để phiên
sau khỏi tiền xử lý lại; checkpoint ghi thẳng lên Drive để không mất khi đứt phiên.

In [ ]:
import os
from pathlib import Path

import torch

# ===================== CHẾ ĐỘ CHẠY =====================
RUN_MODE = "full"        # "debug" = 200 bản ghi / 3 epoch  |  "full" = toàn bộ / 30 epoch

# ===================== ĐƯỜNG DẪN =======================
IS_COLAB = importlib.util.find_spec("google.colab") is not None

if IS_COLAB:
    DRIVE_PROJECT = Path("/content/drive/MyDrive/ACS-ECG-AI")   # <<< SỬA nếu đặt tên khác
    DRIVE_DATA_ZIP = DRIVE_PROJECT / "datasets.zip"             # <<< SỬA nếu tên zip khác
    DATA_ROOT = Path("/content/datasets")      # giải nén ra đĩa local cho nhanh
    WORK_DIR = Path("/content/work")           # tạm, mất khi hết phiên
    PERSIST_DIR = DRIVE_PROJECT / "outputs"    # bền qua các phiên
else:
    LOCAL_DATA_ROOT = r"C:\Users\anhquan\Workspace\AI_THUC_CHIEN\VSF_Projects\ECG-experiment\datasets"                    # <<< SỬA khi chạy máy cá nhân
    DATA_ROOT = Path(LOCAL_DATA_ROOT)
    WORK_DIR = DATA_ROOT.parent / "outputs"
    PERSIST_DIR = WORK_DIR

CACHE_DIR = WORK_DIR / "cache"                     # cache tín hiệu (sinh lại được)
MODEL_DIR = PERSIST_DIR / "models" / "stemi_compare"  # checkpoint (không được mất) -- CHUNG với notebook 03/08, chỉ đọc (ORIG_MODEL_DIR)
DRIVE_CACHE_DIR = (PERSIST_DIR / "cache") if IS_COLAB else None

# ===================== THAM SỐ =========================
SEED = 42
FS, SIGNAL_LEN, NUM_LEADS = 500, 5000, 12      # 12 chuyển đạo × 10 giây @ 500 Hz
BP_LOW, BP_HIGH, BP_ORDER = 0.5, 40.0, 3       # bandpass Butterworth
VAL_SIZE = 0.15                                 # chia theo bệnh nhân 85/15
LR, WEIGHT_DECAY = 1e-3, 1e-4
EARLY_STOP_PATIENCE, LR_PATIENCE = 10, 5
ECE_BINS = 10
TARGET_LABEL = "STEMI"
CLASS_NAME = "STEMI"

# Số bản ghi trong paper (tính trên toàn bộ 19.955); ta chỉ có nhãn cho 90% nên dung sai ±10%
PAPER_COUNTS = {"STEMI": 1513, "NSTEMI": 1274, "UA": 6197}
SANITY_TOL = 0.10

# ===================== SUY RA ==========================
GPU_AVAILABLE = torch.cuda.is_available()
DEVICE = torch.device("cuda" if GPU_AVAILABLE else "cpu")
DEBUG_LIMIT = 200
EPOCHS = 3 if RUN_MODE == "debug" else 30
BATCH_SIZE = 8 if RUN_MODE == "debug" else (64 if GPU_AVAILABLE else 16)
USE_AMP = GPU_AVAILABLE

print("Môi trường :", "Google Colab" if IS_COLAB else "máy cá nhân")
print("Thiết bị   :", DEVICE)
print("Chế độ     :", RUN_MODE, f"({EPOCHS} epoch, batch {BATCH_SIZE})")
print("Nhãn đích  :", TARGET_LABEL)

# ============ K-FOLD + HIỆU CHUẨN (notebook 08) ============
ORIG_MODEL_DIR = MODEL_DIR                 # checkpoint của notebook 03 — CHỈ đọc, để inference lại
# "_holdout": thử nghiệm này train trên 85% (đã trích 15% làm TEST riêng ở mục 12b) -- khác dữ liệu
# với thử nghiệm K-Fold cũ (train trên toàn bộ 17.960 bản ghi). Đặt thư mục/tên file riêng để
# không ghi đè checkpoint và OOF của thử nghiệm cũ.
MODEL_DIR = PERSIST_DIR / "models" / "stemi_expanded_holdout"
OOF_DIR = PERSIST_DIR / "oof_expanded"      # thư mục RIÊNG với notebook 08 -- tránh ghi đè OOF cache

K_FOLDS = 5
N_REPEATS = 1          # 2 -> 10 cặp, Wilcoxon HAI phía mới đủ lực (xem mục 16). Gấp đôi chi phí GPU.
# CHỈ 5 kiến trúc MỚI (notebook 08 đã có kết quả K-Fold cho PlainCNN/XResNet1D/ConvNeXtV2_1D/
# AiTiAMI/ResNet1D/SEResNet1D/CNN+BiLSTM/InceptionTime1D -- không train lại 8 model đó ở đây).
CANDIDATES = ["S4D1D", "Mamba2Lite1D", "ECGMamba1D", "OMICNN1D", "OSCNN1D"]
SPEC_TARGET = 0.85     # điểm vận hành chung để so matched-threshold
RULE_OUT_NPV = 0.99    # ngưỡng rule-out lâm sàng
N_BOOTSTRAP = 1000
RUN_SWEEP = False      # True = chạy mục 20 (sweep hyperparameter), tốn thêm nhiều giờ GPU

N_FOLDS_RUN = 1 if RUN_MODE == "debug" else K_FOLDS   # debug chỉ chạy fold đầu

print("Ứng viên   :", ", ".join(CANDIDATES))
print("K-Fold     :", f"{K_FOLDS} fold × {N_REPEATS} lần lặp, chạy {N_FOLDS_RUN} fold ở chế độ {RUN_MODE}")
print("Checkpoint :", MODEL_DIR)

### 2b. Chuẩn bị dữ liệu

Trên máy cá nhân cell này chỉ tạo thư mục. Trên Colab nó mount Drive (sẽ hiện popup xin quyền —
đây là thao tác thủ công duy nhất), copy `datasets.zip` về `/content` rồi giải nén.

Phải đi vòng qua zip vì bộ dữ liệu có **59.867 file**, mà Drive tính mỗi lần mở file là một lệnh
gọi mạng — đọc trực tiếp từ Drive sẽ chậm hơn hàng chục lần.

In [ ]:
import shutil
import time
import zipfile


def _locate_data_root(search_root: Path) -> Path:
    """Tìm thư mục thật chứa dữ liệu, chịu được việc zip có bọc thêm lớp thư mục."""
    wanted = {"csv", "row_data", "raw_data", "med_data"}
    best, best_score = search_root, -1
    for cand in [search_root, *[p for p in search_root.rglob("*") if p.is_dir()]]:
        try:
            names = {c.name.lower() for c in cand.iterdir() if c.is_dir()}
        except OSError:
            continue
        if len(names & wanted) > best_score:
            best, best_score = cand, len(names & wanted)
        if best_score >= 2:
            break
    return best


def stage_colab_data() -> Path:
    marker = DATA_ROOT / ".staged_ok"
    if marker.exists():
        real = Path(marker.read_text().strip())
        print("Dữ liệu đã sẵn sàng:", real)
        return real

    if not DRIVE_DATA_ZIP.exists():
        có_gì = sorted(p.name for p in DRIVE_PROJECT.iterdir())[:40]
        raise FileNotFoundError(
            f"Không thấy {DRIVE_DATA_ZIP}\nTrong {DRIVE_PROJECT} hiện có: {có_gì}"
        )

    local_zip = Path("/content/_dataset.zip")
    size = DRIVE_DATA_ZIP.stat().st_size
    if not (local_zip.exists() and local_zip.stat().st_size == size):
        print(f"Copy zip {size / 1024 ** 3:.2f} GB từ Drive ...")
        t0 = time.time()
        shutil.copy2(DRIVE_DATA_ZIP, local_zip)
        print(f"  {time.time() - t0:.0f}s")

    extract_to = Path("/content/_extract")
    if extract_to.exists():
        shutil.rmtree(extract_to)
    print("Giải nén ...")
    t0 = time.time()
    with zipfile.ZipFile(local_zip) as zf:
        zf.extractall(extract_to)
    print(f"  {time.time() - t0:.0f}s")

    real = _locate_data_root(extract_to)
    real.mkdir(parents=True, exist_ok=True)
    (real / ".staged_ok").write_text(str(real))
    return real


if IS_COLAB:
    if str(DRIVE_PROJECT).startswith(("http://", "https://", "www.")):
        raise ValueError(
            "DRIVE_PROJECT đang là URL chia sẻ Drive. Phải là đường dẫn sau khi mount:\n"
            '    DRIVE_PROJECT = Path("/content/drive/MyDrive/ACS-ECG-AI")'
        )
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    if not DRIVE_PROJECT.exists():
        my = Path("/content/drive/MyDrive")
        có_gì = sorted(p.name for p in my.iterdir() if p.is_dir())[:40] if my.exists() else []
        raise FileNotFoundError(
            f"Không thấy {DRIVE_PROJECT}\nCác thư mục trong MyDrive: {có_gì}"
        )

for d in [CACHE_DIR, MODEL_DIR] + ([DRIVE_CACHE_DIR] if DRIVE_CACHE_DIR else []):
    d.mkdir(parents=True, exist_ok=True)

if IS_COLAB:
    DATA_ROOT = stage_colab_data()

print("DATA_ROOT:", DATA_ROOT, "| tồn tại:", DATA_ROOT.exists())

## 3. Seed & GPU

In [ ]:
import random

import numpy as np

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Input cố định (12 × 5000) nên cuDNN chọn thuật toán nhanh nhất một lần rồi giữ nguyên.
# Đánh đổi: epoch đầu chậm hơn vài giây, và kết quả không tái lập chính xác 100%.
torch.backends.cudnn.benchmark = GPU_AVAILABLE

print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if GPU_AVAILABLE else "không có")
if IS_COLAB and not GPU_AVAILABLE:
    print("\n" + "!" * 70)
    print("!! COLAB KHÔNG CÓ GPU -> Runtime > Change runtime type > GPU, rồi Run all lại.")
    print("!" * 70)

## 4. Dữ liệu và nhãn

Nhãn chỉ có trong `train.csv` (phần 90% chính thức). `test.csv` là tập test ẩn, không có cột nhãn
và **không bao giờ được đọc** trong notebook này.

Sanity check đối chiếu số ca với paper. Dung sai ±10% chứ không phải ±2%, vì số của paper tính trên
toàn bộ 19.955 bản ghi còn ta chỉ đếm được trên 90% có nhãn.

In [ ]:
import pandas as pd

pd.set_option("display.width", 200)


def find_dir(root: Path, names):
    wanted = {n.lower() for n in names}
    for c in sorted(root.iterdir()):
        if c.is_dir() and c.name.lower() in wanted:
            return c
    for c in root.rglob("*"):
        if c.is_dir() and c.name.lower() in wanted:
            return c
    return None


RAW_DIR = find_dir(DATA_ROOT, ["row_data", "raw_data"])
CSV_DIR = find_dir(DATA_ROOT, ["CSV", "csv"])
assert RAW_DIR and CSV_DIR, f"Không thấy row_data/ hoặc CSV/ trong {DATA_ROOT}"

df_raw = pd.read_csv(CSV_DIR / "train.csv")
df_raw["record_stem"] = df_raw["ecg_row_record"].astype(str).str.replace(".dat", "", regex=False)

COL_PATIENT, COL_AGE, COL_GENDER = "Patient_id", "age", "gender"
print(f"{len(df_raw):,} bản ghi | {df_raw[COL_PATIENT].nunique():,} bệnh nhân | "
      f"{len(list(RAW_DIR.glob('*.dat'))):,} file .dat")

# --- Sanity check ---
print(f"\n{'nhãn':<8}{'đếm được':>11}{'paper':>9}{'lệch':>9}")
ok_all = True
for name, expected in PAPER_COUNTS.items():
    got = int(df_raw[name].sum())
    delta = (got - expected) / expected
    ok_all &= abs(delta) <= SANITY_TOL
    print(f"{name:<8}{got:>11,}{expected:>9,}{delta:>8.1%}   {'OK' if abs(delta) <= SANITY_TOL else 'LỆCH'}")
if not ok_all:
    print("\n!! CẢNH BÁO: số ca lệch quá ±10% so với paper — kiểm tra lại nguồn dữ liệu.")

# --- Nhãn đích ---
df_all = df_raw.copy()
df_all[TARGET_LABEL] = df_all["STEMI"].astype(int)
print("Nhãn = cột STEMI (lấy trực tiếp)")

n_pos = int(df_all[TARGET_LABEL].sum())
print(f"\nNhãn {TARGET_LABEL}: {n_pos:,} dương ({n_pos / len(df_all):.2%}) / "
      f"{len(df_all) - n_pos:,} âm")

## 5. Lấy mẫu khi chạy thử

Ở `RUN_MODE="debug"` lấy 200 bản ghi **stratified** theo nhãn, để tập mẫu không rơi vào toàn 0
hoặc toàn 1 (khi đó mọi metric sẽ vô định).

In [ ]:
from sklearn.model_selection import train_test_split

if RUN_MODE == "debug" and len(df_all) > DEBUG_LIMIT:
    df, _ = train_test_split(df_all, train_size=DEBUG_LIMIT,
                             stratify=df_all[TARGET_LABEL], random_state=SEED)
    df = df.reset_index(drop=True)
else:
    df = df_all.reset_index(drop=True)

n_pos = int(df[TARGET_LABEL].sum())
print(f"Dùng {len(df):,} bản ghi | {n_pos:,} dương ({n_pos / len(df):.2%})")
assert 0 < n_pos < len(df), "Tập chỉ có một lớp — không train được."

## 7. Tiền xử lý và cache

Với mỗi bản ghi: đọc WFDB → NaN→0 → cắt/đệm về đúng 5000 mẫu → bandpass Butterworth 0,5–40 Hz
(`filtfilt`, zero-phase). **Chuẩn hoá z-score làm sau**, ở bước 9, với mean/std tính riêng trên tập
train để không rò rỉ thống kê của val sang train.

Lọc lại toàn bộ ở mỗi epoch là nút thắt cổ chai, nên tín hiệu đã lọc được cache **một lần** vào
memmap `float16` (~2,2 GB cho 17.960 bản ghi). Cache nối tiếp được nếu bị ngắt giữa chừng, và trên
Colab được sao lưu lên Drive.

> **Hai bản ghi lỗi trong dữ liệu gốc.** `03228` và `14262` có file `.dat` chỉ chứa 3.500/5.000 mẫu,
> khiến `wfdb.rdrecord()` báo lỗi. Hàm dưới bắt lỗi đó, đọc phần thực có rồi để bước zero-pad bù nốt.

In [ ]:
import hashlib
import json

from scipy.signal import butter, filtfilt

_B, _A = butter(BP_ORDER, [BP_LOW / (FS / 2), BP_HIGH / (FS / 2)], btype="band")
TRUNCATED = []


def load_signal(stem: str) -> np.ndarray:
    path = str(RAW_DIR / stem)
    try:
        rec = wfdb.rdrecord(path)
    except ValueError:                     # .dat ngắn hơn header khai báo
        n_sig = int((RAW_DIR / f"{stem}.hea").read_text().splitlines()[0].split()[1])
        n = (RAW_DIR / f"{stem}.dat").stat().st_size // (n_sig * 2)
        rec = wfdb.rdrecord(path, sampto=n)
        TRUNCATED.append(stem)
    return np.asarray(rec.p_signal, dtype=np.float32).T


def preprocess(sig: np.ndarray) -> np.ndarray:
    sig = np.nan_to_num(sig, nan=0.0, posinf=0.0, neginf=0.0)
    if sig.shape[1] < SIGNAL_LEN:
        sig = np.pad(sig, ((0, 0), (0, SIGNAL_LEN - sig.shape[1])))
    return np.ascontiguousarray(filtfilt(_B, _A, sig[:, :SIGNAL_LEN], axis=1), dtype=np.float32)


records = df["record_stem"].tolist()
_hash = hashlib.md5("|".join(records).encode()).hexdigest()[:8]
CACHE_NPY = CACHE_DIR / f"sig_{RUN_MODE}_n{len(df)}_{_hash}.npy"
CACHE_META = CACHE_DIR / f"sig_{RUN_MODE}_n{len(df)}_{_hash}.meta.json"


def _complete(meta_p: Path, npy_p: Path) -> bool:
    if not (meta_p.exists() and npy_p.exists()):
        return False
    try:
        return json.loads(meta_p.read_text()).get("n_done", 0) >= len(records)
    except (OSError, ValueError):
        return False


def build_cache() -> np.ndarray:
    # Colab: thử lấy lại cache của phiên trước từ Drive
    if DRIVE_CACHE_DIR and not _complete(CACHE_META, CACHE_NPY):
        d_npy, d_meta = DRIVE_CACHE_DIR / CACHE_NPY.name, DRIVE_CACHE_DIR / CACHE_META.name
        if _complete(d_meta, d_npy):
            print(f"Lấy cache từ Drive ({d_npy.stat().st_size / 1024 ** 3:.2f} GB) ...")
            t0 = time.time()
            shutil.copy2(d_npy, CACHE_NPY)
            shutil.copy2(d_meta, CACHE_META)
            print(f"  {time.time() - t0:.0f}s — bỏ qua tiền xử lý")

    if _complete(CACHE_META, CACHE_NPY):
        print("Cache đã đầy đủ:", CACHE_NPY.name)
        return np.load(CACHE_NPY, mmap_mode="r")

    meta = json.loads(CACHE_META.read_text()) if CACHE_META.exists() else {}
    start = int(meta.get("n_done", 0)) if CACHE_NPY.exists() else 0
    if start:
        arr = np.lib.format.open_memmap(CACHE_NPY, mode="r+")
        print(f"Build tiếp từ {start}/{len(records)}")
    else:
        print(f"Build cache {len(records):,} bản ghi "
              f"(~{len(records) * NUM_LEADS * SIGNAL_LEN * 2 / 1024 ** 3:.2f} GB)")
        arr = np.lib.format.open_memmap(CACHE_NPY, mode="w+", dtype=np.float16,
                                        shape=(len(records), NUM_LEADS, SIGNAL_LEN))

    t0 = time.time()
    step = max(1, len(records) // 8)
    for i in range(start, len(records)):
        arr[i] = preprocess(load_signal(records[i])).astype(np.float16)
        if (i + 1) % step == 0 or i + 1 == len(records):
            arr.flush()
            CACHE_META.write_text(json.dumps({"n_done": i + 1}))
            el = max(time.time() - t0, 1e-6)
            done = i + 1 - start
            print(f"  {i + 1:>6,}/{len(records):,}  {done / el:5.0f} rec/s  "
                  f"ETA {(len(records) - i - 1) / (done / el):4.0f}s")
    del arr
    print(f"Xong trong {time.time() - t0:.0f}s")

    if DRIVE_CACHE_DIR:
        print("Sao lưu cache lên Drive ...")
        shutil.copy2(CACHE_NPY, DRIVE_CACHE_DIR / CACHE_NPY.name)
        shutil.copy2(CACHE_META, DRIVE_CACHE_DIR / CACHE_META.name)
    return np.load(CACHE_NPY, mmap_mode="r")


CACHE = build_cache()
print("Cache:", CACHE.shape, CACHE.dtype)
if TRUNCATED:
    print(f"Bản ghi bị cắt ngắn, đã zero-pad: {TRUNCATED}")

## 9. Chia train/val theo bệnh nhân

Bắt buộc chia theo `Patient_id`: một bệnh nhân có thể có nhiều ECG trong 7 ngày trước DSA, chia theo
bản ghi sẽ khiến cùng một người xuất hiện ở cả train lẫn val.

Stratify ở **cấp bệnh nhân** (lấy `max` nhãn của người đó) — cần thiết vì có bệnh nhân mang nhãn
không đồng nhất giữa các bản ghi của chính mình (xem mục 6).

Ngay sau khi chia, mean/std cho z-score được tính **chỉ trên các bản ghi thuộc train**.

In [ ]:
pat = df.groupby(COL_PATIENT)[TARGET_LABEL].max().reset_index()
strat = pat[TARGET_LABEL] if pat[TARGET_LABEL].value_counts().min() >= 2 else None
pat_tr, pat_va = train_test_split(pat, test_size=VAL_SIZE, stratify=strat, random_state=SEED)

train_idx = df.index[df[COL_PATIENT].isin(set(pat_tr[COL_PATIENT]))].to_numpy()
val_idx = df.index[df[COL_PATIENT].isin(set(pat_va[COL_PATIENT]))].to_numpy()
y = df[TARGET_LABEL].values.astype(np.float32)

assert not (set(pat_tr[COL_PATIENT]) & set(pat_va[COL_PATIENT])), "RÒ RỈ: bệnh nhân ở cả 2 tập"
print("Giao nhau patient_id: RỖNG -> OK\n")

display(pd.DataFrame({
    "tập": ["train", "val"],
    "bệnh nhân": [len(pat_tr), len(pat_va)],
    "bản ghi": [len(train_idx), len(val_idx)],
    "dương": [int(y[train_idx].sum()), int(y[val_idx].sum())],
    "tỷ lệ dương": [f"{y[train_idx].mean():.2%}", f"{y[val_idx].mean():.2%}"],
}))


def norm_stats(idx, chunk=256):
    order = np.sort(np.asarray(idx))
    s = np.zeros(NUM_LEADS)
    ss = np.zeros(NUM_LEADS)
    cnt = 0
    for i in range(0, len(order), chunk):
        b = np.asarray(CACHE[order[i:i + chunk]], dtype=np.float64)
        s += b.sum(axis=(0, 2))
        ss += (b ** 2).sum(axis=(0, 2))
        cnt += b.shape[0] * b.shape[2]
    m = s / cnt
    return m.astype(np.float32), np.sqrt(np.maximum(ss / cnt - m ** 2, 1e-12)).astype(np.float32)


LEAD_MEAN, LEAD_STD = norm_stats(train_idx)
print(f"\nz-score fit trên {len(train_idx):,} bản ghi TRAIN")

## 10. Dataset & DataLoader

In [ ]:
from torch.utils.data import DataLoader, Dataset


class ECGDataset(Dataset):
    """Khác notebook 03 đúng một chỗ: nhận mean/std làm tham số.

    z-score phải fit lại trên tập train CỦA TỪNG FOLD, không dùng chung một bộ
    thống kê toàn cục được — nếu không thì thống kê của fold này rò rỉ sang fold kia.
    Mặc định vẫn là LEAD_MEAN/LEAD_STD của lần chia 85/15 gốc, để mục 17
    (nạp lại checkpoint cũ) chuẩn hoá y hệt lúc các model đó được huấn luyện.
    """

    def __init__(self, indices, labels, mean=None, std=None):
        self.indices = np.asarray(indices)
        self.labels = np.asarray(labels, dtype=np.float32)
        self.mean = (LEAD_MEAN if mean is None else mean).reshape(-1, 1)
        self.std = (LEAD_STD if std is None else std).reshape(-1, 1)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        x = np.asarray(CACHE[self.indices[i]], dtype=np.float32)
        return torch.from_numpy((x - self.mean) / self.std), torch.tensor(self.labels[i])


# Windows tạo worker bằng spawn (mỗi worker nạp lại cả notebook) nên để 0; Linux dùng fork -> 2
NUM_WORKERS = 0 if os.name == "nt" else 2
_extra = dict(persistent_workers=True, prefetch_factor=4) if NUM_WORKERS else {}


def make_loaders(tr_idx, va_idx, mean=None, std=None):
    ds_tr = ECGDataset(tr_idx, y[tr_idx], mean, std)
    ds_va = ECGDataset(va_idx, y[va_idx], mean, std)
    tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=NUM_WORKERS, pin_memory=GPU_AVAILABLE, **_extra)
    va = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=GPU_AVAILABLE, **_extra)
    return tr, va


# Loader của lần chia 85/15 GỐC — chỉ dùng ở mục 17 để chấm lại checkpoint của notebook 03
train_loader, val_loader = make_loaders(train_idx, val_idx)

xb, yb = next(iter(val_loader))
print(f"batch: {tuple(xb.shape)} {xb.dtype} | nhãn {tuple(yb.shape)} | "
      f"{len(train_loader)} batch train, {len(val_loader)} batch val (split 85/15 gốc)")

## 11. Hàm tính metric

`compute_metrics` an toàn với tập nhỏ: khi chỉ có một lớp hoặc mẫu số bằng 0, trả `nan` thay vì
ném lỗi hay trả 0 gây hiểu nhầm.

`per_class_report` cho bảng tách riêng **lớp âm tính (0)** và **lớp dương tính (1)** — mỗi lớp có
Precision/Recall/F1/Support riêng, kèm hai dòng trung bình macro và weighted.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (average_precision_score, brier_score_loss, confusion_matrix,
                             precision_recall_fscore_support, roc_auc_score)

plt.rcParams["figure.dpi"] = 110


def _div(a, b):
    return float(a) / float(b) if b else float("nan")


def compute_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int).ravel()
    y_prob = np.asarray(y_prob, dtype=np.float64).ravel()
    y_pred = (y_prob >= threshold).astype(int)
    two = len(np.unique(y_true)) == 2
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "auroc": roc_auc_score(y_true, y_prob) if two else float("nan"),
        "auprc": average_precision_score(y_true, y_prob) if two else float("nan"),
        "sensitivity": _div(tp, tp + fn), "specificity": _div(tn, tn + fp),
        "ppv": _div(tp, tp + fp), "npv": _div(tn, tn + fn),
        "f1": _div(2 * tp, 2 * tp + fp + fn),
        "accuracy": _div(tp + tn, tp + tn + fp + fn),
        "brier": float(brier_score_loss(y_true, y_prob)) if two else float("nan"),
        "threshold": float(threshold),
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
        "n": int(len(y_true)), "n_pos": int(y_true.sum()),
    }


def per_class_report(y_true, y_prob, threshold):
    """Bảng metric TÁCH RIÊNG cho lớp 0 và lớp 1, kiểu classification_report của sklearn."""
    y_true = np.asarray(y_true).astype(int).ravel()
    y_pred = (np.asarray(y_prob).ravel() >= threshold).astype(int)
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1], zero_division=0)
    rows = [
        {"Class": f"Negative (0) — không {CLASS_NAME}", "Precision": p[0], "Recall": r[0],
         "F1": f[0], "Support": int(s[0])},
        {"Class": f"Positive (1) — {CLASS_NAME}", "Precision": p[1], "Recall": r[1],
         "F1": f[1], "Support": int(s[1])},
    ]
    w = s / s.sum()
    rows.append({"Class": "macro avg", "Precision": p.mean(), "Recall": r.mean(),
                 "F1": f.mean(), "Support": int(s.sum())})
    rows.append({"Class": "weighted avg", "Precision": float(p @ w), "Recall": float(r @ w),
                 "F1": float(f @ w), "Support": int(s.sum())})
    return pd.DataFrame(rows).set_index("Class")


def style_df(df, pct_cols):
    fmt = {c: "{:.4f}" for c in pct_cols}
    fmt.update({c: "{:,d}" for c in df.columns if c not in pct_cols and df[c].dtype.kind in "iu"})
    try:
        return df.style.format(fmt, na_rep="n/a").background_gradient(
            cmap="Blues", vmin=0, vmax=1, subset=[c for c in pct_cols if c in df.columns])
    except Exception:
        return df.round(4)


print("self-test:", {k: round(v, 3) for k, v in
                     compute_metrics([0, 0, 1, 1], [.1, .4, .35, .8]).items()
                     if k in ("auroc", "f1", "sensitivity")})

## 12. Mười lăm kiến trúc (định nghĩa) -- chỉ 5 kiến trúc mới được đưa vào K-Fold

Tất cả nhận input `(batch, 12, 5000)` và trả về **một logit** cho mỗi mẫu, nên có thể thay thế nhau
trong đúng một vòng lặp huấn luyện.

Tất cả đều hạ chiều thời gian khá mạnh (5000 → vài chục/vài trăm) trước khi gộp, vì tín hiệu 10
giây ở 500 Hz là quá dài để giữ nguyên độ phân giải qua nhiều tầng — kể cả `S4D1D`/`Mamba2Lite1D`,
vốn về lý thuyết xử lý được chuỗi dài trực tiếp, cũng được hạ mẫu trước bằng CNN-stem để tiết kiệm
thời gian huấn luyện.

> ⚠️ **`AiTiAMI` là tái hiện theo mô tả kiến trúc, không phải model/trọng số gốc.** Lee *et al.*
> (*Eur Heart J* 2025, ehaf004 — nghiên cứu ROMIAE) chỉ mô tả AiTiAMI v1.00.00 của Medical AI Co.
> là "an advanced algorithm... built on a residual neural network", đầu vào 500Hz 12 chuyển đạo
> thô, đầu ra điểm xác suất AMI 0-100. Không có chi tiết số khối, độ sâu, hay trọng số công khai —
> class `AiTiAMI` dưới đây là một ResNet1D sâu/rộng hơn `ResNet1D` sẵn có, kernel hẹp hơn (5 thay
> vì 7) để bắt chi tiết QRS/ST mịn hơn, head kết hợp avg+max-pool. Kết quả của nó **không thể**
> dùng để so sánh trực tiếp với AUROC 0,971 (STEMI) mà bài báo báo cáo cho AiTiAMI thật — bài báo
> đó đánh giá trên dữ liệu và quần thể bệnh nhân Hàn Quốc hoàn toàn khác.

> ⚠️ **`S4D1D`, `Mamba2Lite1D`, `ECGMamba1D`, `OMICNN1D`, `OSCNN1D` đều là cài đặt lại/xấp xỉ,
> không phải mã nguồn hay trọng số gốc của bất kỳ công bố/sản phẩm nào** — xem bảng chi tiết ở mục
> giới thiệu notebook (đầu file). Không nên trích dẫn kết quả của các model này như thể là kết quả
> của Mamba-2/S4/ECGMamba/Queen-of-Hearts/Omni-Scale-CNN thật.

> ℹ️ Toàn bộ 15 class ở đây (10 từ notebook 03/08 + 5 model mới) đều được định nghĩa để mục 17(a)
> có thể nạp checkpoint cũ và inference lại, nhưng `CANDIDATES` ở mục 2 chỉ liệt kê **5 model mới**
> — chỉ chúng mới được huấn luyện K-Fold trong notebook này.

In [ ]:
import math
import torch.nn as nn


# ---------------------------------------------------------------- 1. PlainCNN
class PlainCNN(nn.Module):
    """Mốc sàn: Conv-BN-ReLU-MaxPool xếp chồng, không có gì đặc biệt."""

    def __init__(self, channels=(32, 64, 128, 256)):
        super().__init__()
        layers, c_in = [], NUM_LEADS
        for c_out in channels:
            layers += [nn.Conv1d(c_in, c_out, 7, padding=3, bias=False),
                       nn.BatchNorm1d(c_out), nn.ReLU(inplace=True), nn.MaxPool1d(4)]
            c_in = c_out
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Dropout(0.3), nn.Linear(c_in, 1))

    def forward(self, x):
        return self.head(self.features(x)).squeeze(-1)


# ---------------------------------------------------------------- 2. ResNet1D
class ResidualBlock1D(nn.Module):
    def __init__(self, c_in, c_out, k=7, stride=2, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(c_in, c_out, k, stride, k // 2, bias=False)
        self.bn1 = nn.BatchNorm1d(c_out)
        self.conv2 = nn.Conv1d(c_out, c_out, k, 1, k // 2, bias=False)
        self.bn2 = nn.BatchNorm1d(c_out)
        self.drop = nn.Dropout(dropout)
        self.relu = nn.ReLU(inplace=True)
        self.short = (nn.Identity() if (stride == 1 and c_in == c_out)
                      else nn.Sequential(nn.Conv1d(c_in, c_out, 1, stride, bias=False),
                                         nn.BatchNorm1d(c_out)))

    def forward(self, x):
        idt = self.short(x)
        out = self.drop(self.relu(self.bn1(self.conv1(x))))
        return self.relu(self.bn2(self.conv2(out)) + idt)


class ResNet1D(nn.Module):
    """Skip connection giúp gradient đi xuyên qua mạng sâu."""

    def __init__(self, channels=(32, 64, 128, 256), dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(NUM_LEADS, 32, 15, 2, 7, bias=False),
                                  nn.BatchNorm1d(32), nn.ReLU(inplace=True), nn.MaxPool1d(2))
        blocks, c_in = [], 32
        for c_out in channels:
            blocks.append(ResidualBlock1D(c_in, c_out, stride=2))
            c_in = c_out
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Linear(c_in, 64), nn.ReLU(inplace=True),
                                  nn.Dropout(dropout), nn.Linear(64, 1))

    def forward(self, x):
        return self.head(self.blocks(self.stem(x))).squeeze(-1)


# ---------------------------------------------------------------- 3. InceptionTime1D
class InceptionModule(nn.Module):
    """Nhiều độ dài kernel song song -> bắt được cả sóng nhanh lẫn biến thiên chậm."""

    def __init__(self, c_in, n_filters=32, kernels=(39, 19, 9), bottleneck=32):
        super().__init__()
        self.bottleneck = nn.Conv1d(c_in, bottleneck, 1, bias=False)
        self.convs = nn.ModuleList(
            [nn.Conv1d(bottleneck, n_filters, k, padding=k // 2, bias=False) for k in kernels])
        self.pool_conv = nn.Sequential(nn.MaxPool1d(3, stride=1, padding=1),
                                       nn.Conv1d(c_in, n_filters, 1, bias=False))
        self.bn = nn.BatchNorm1d(n_filters * (len(kernels) + 1))
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        b = self.bottleneck(x)
        return self.relu(self.bn(torch.cat([c(b) for c in self.convs] + [self.pool_conv(x)], 1)))


class InceptionTime1D(nn.Module):
    def __init__(self, n_filters=32, depth=6):
        super().__init__()
        # hạ tần số ngay từ đầu, nếu không 5000 mẫu × nhiều kênh sẽ quá nặng
        self.stem = nn.Sequential(nn.Conv1d(NUM_LEADS, 32, 15, 4, 7, bias=False),
                                  nn.BatchNorm1d(32), nn.ReLU(inplace=True))
        c_out = n_filters * 4
        self.blocks = nn.ModuleList()
        self.shortcuts = nn.ModuleList()
        self.pools = nn.ModuleList()
        c_in = 32
        res_c = 32          # số kênh của tensor dùng làm residual, KHÁC c_in của khối hiện tại
        for d in range(depth):
            self.blocks.append(InceptionModule(c_in, n_filters))
            # residual mỗi 3 khối, kèm hạ chiều thời gian
            if d % 3 == 2:
                self.shortcuts.append(nn.Sequential(nn.Conv1d(res_c, c_out, 1, bias=False),
                                                    nn.BatchNorm1d(c_out)))
                self.pools.append(nn.MaxPool1d(4))
                res_c = c_out
            else:
                self.shortcuts.append(None)
                self.pools.append(None)
            c_in = c_out
        self.relu = nn.ReLU(inplace=True)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Dropout(0.3), nn.Linear(c_out, 1))

    def forward(self, x):
        x = self.stem(x)
        res = x
        for blk, short, pool in zip(self.blocks, self.shortcuts, self.pools):
            x = blk(x)
            if short is not None:
                x = self.relu(x + short(res))
                x = pool(x)
                res = x
        return self.head(x).squeeze(-1)


# ---------------------------------------------------------------- 4. CNN + BiLSTM
class CNNBiLSTM(nn.Module):
    """CNN rút đặc trưng cục bộ, BiLSTM mô hình hoá quan hệ theo thời gian giữa các nhịp."""

    def __init__(self, hidden=128):
        super().__init__()
        layers, c_in = [], NUM_LEADS
        for c_out in (32, 64, 128):
            layers += [nn.Conv1d(c_in, c_out, 7, padding=3, bias=False),
                       nn.BatchNorm1d(c_out), nn.ReLU(inplace=True), nn.MaxPool1d(4)]
            c_in = c_out
        self.cnn = nn.Sequential(*layers)                       # 5000 -> ~78
        self.lstm = nn.LSTM(c_in, hidden, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(hidden * 2, 1))

    def forward(self, x):
        h = self.cnn(x).transpose(1, 2)                          # (B, T, C)
        out, _ = self.lstm(h)
        return self.head(out.mean(dim=1)).squeeze(-1)            # gộp trung bình theo thời gian

# ---------------------------------------------------------------- 5. SEResNet1D
class SEBlock1D(nn.Module):
    """Squeeze-excitation: học trọng số quan trọng theo từng kênh (feature map)."""

    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(nn.Linear(channels, hidden), nn.ReLU(inplace=True),
                                nn.Linear(hidden, channels), nn.Sigmoid())

    def forward(self, x):
        w = self.fc(self.pool(x).squeeze(-1)).unsqueeze(-1)
        return x * w


class SEResidualBlock1D(nn.Module):
    def __init__(self, c_in, c_out, k=7, stride=2, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(c_in, c_out, k, stride, k // 2, bias=False)
        self.bn1 = nn.BatchNorm1d(c_out)
        self.conv2 = nn.Conv1d(c_out, c_out, k, 1, k // 2, bias=False)
        self.bn2 = nn.BatchNorm1d(c_out)
        self.se = SEBlock1D(c_out)
        self.drop = nn.Dropout(dropout)
        self.relu = nn.ReLU(inplace=True)
        self.short = (nn.Identity() if (stride == 1 and c_in == c_out)
                      else nn.Sequential(nn.Conv1d(c_in, c_out, 1, stride, bias=False),
                                         nn.BatchNorm1d(c_out)))

    def forward(self, x):
        idt = self.short(x)
        out = self.drop(self.relu(self.bn1(self.conv1(x))))
        out = self.se(self.bn2(self.conv2(out)))
        return self.relu(out + idt)


class SEResNet1D(nn.Module):
    """ResNet1D + squeeze-excitation sau mỗi khối."""

    def __init__(self, channels=(32, 64, 128, 256), dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(NUM_LEADS, 32, 15, 2, 7, bias=False),
                                  nn.BatchNorm1d(32), nn.ReLU(inplace=True), nn.MaxPool1d(2))
        blocks, c_in = [], 32
        for c_out in channels:
            blocks.append(SEResidualBlock1D(c_in, c_out, stride=2))
            c_in = c_out
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Linear(c_in, 64), nn.ReLU(inplace=True),
                                  nn.Dropout(dropout), nn.Linear(64, 1))

    def forward(self, x):
        return self.head(self.blocks(self.stem(x))).squeeze(-1)


# ---------------------------------------------------------------- 6. Transformer1D
class Transformer1D(nn.Module):
    """Cắt tín hiệu thành patch theo thời gian, học vị trí, qua self-attention."""

    def __init__(self, patch_len=250, d_model=128, n_heads=4, n_layers=4, dropout=0.1):
        super().__init__()
        assert SIGNAL_LEN % patch_len == 0
        n_patches = SIGNAL_LEN // patch_len
        self.patch_len = patch_len
        self.proj = nn.Linear(NUM_LEADS * patch_len, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, d_model))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        layer = nn.TransformerEncoderLayer(d_model, n_heads, dim_feedforward=d_model * 4,
                                           dropout=dropout, batch_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(layer, n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(d_model, 1))

    def forward(self, x):
        b = x.shape[0]
        patches = x.unfold(-1, self.patch_len, self.patch_len)          # (B, 12, n_patches, patch_len)
        patches = patches.permute(0, 2, 1, 3).reshape(b, patches.shape[2], -1)
        tok = self.proj(patches)
        tok = torch.cat([self.cls_token.expand(b, -1, -1), tok], dim=1) + self.pos_embed
        out = self.norm(self.encoder(tok))
        return self.head(out[:, 0]).squeeze(-1)


# ---------------------------------------------------------------- 7. TCN1D
class TCNBlock1D(nn.Module):
    """Conv1D giãn nở (dilated), residual — receptive field rộng mà giữ độ phân giải thời gian."""

    def __init__(self, c_in, c_out, k=7, dilation=1, dropout=0.1):
        super().__init__()
        pad = (k - 1) * dilation // 2
        self.conv1 = nn.Conv1d(c_in, c_out, k, padding=pad, dilation=dilation, bias=False)
        self.bn1 = nn.BatchNorm1d(c_out)
        self.conv2 = nn.Conv1d(c_out, c_out, k, padding=pad, dilation=dilation, bias=False)
        self.bn2 = nn.BatchNorm1d(c_out)
        self.drop = nn.Dropout(dropout)
        self.relu = nn.ReLU(inplace=True)
        self.short = nn.Identity() if c_in == c_out else nn.Conv1d(c_in, c_out, 1, bias=False)

    def forward(self, x):
        out = self.drop(self.relu(self.bn1(self.conv1(x))))
        out = self.bn2(self.conv2(out))
        return self.relu(out + self.short(x))


class TCN1D(nn.Module):
    def __init__(self, channels=(32, 64, 128, 128, 256, 256), dropout=0.15):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(NUM_LEADS, 32, 15, 2, 7, bias=False),
                                  nn.BatchNorm1d(32), nn.ReLU(inplace=True), nn.MaxPool1d(2))
        blocks, c_in = [], 32
        for i, c_out in enumerate(channels):
            blocks.append(TCNBlock1D(c_in, c_out, dilation=2 ** i, dropout=dropout))
            if i % 2 == 1:                       # hạ chiều thời gian định kỳ, không hạ mỗi lớp
                blocks.append(nn.MaxPool1d(2))
            c_in = c_out
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Dropout(0.3), nn.Linear(c_in, 1))

    def forward(self, x):
        return self.head(self.blocks(self.stem(x))).squeeze(-1)


# ---------------------------------------------------------------- 8. XResNet1D
class XResBlock1D(nn.Module):
    """Nhánh tắt kiểu xResNet: AvgPool + 1x1 conv thay vì strided conv khi downsample."""

    def __init__(self, c_in, c_out, k=7, stride=2, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(c_in, c_out, k, stride, k // 2, bias=False)
        self.bn1 = nn.BatchNorm1d(c_out)
        self.conv2 = nn.Conv1d(c_out, c_out, k, 1, k // 2, bias=False)
        self.bn2 = nn.BatchNorm1d(c_out)
        self.drop = nn.Dropout(dropout)
        self.act = nn.SiLU(inplace=True)
        if stride == 1 and c_in == c_out:
            self.short = nn.Identity()
        else:
            self.short = nn.Sequential(
                nn.AvgPool1d(stride, ceil_mode=True), nn.Conv1d(c_in, c_out, 1, bias=False),
                nn.BatchNorm1d(c_out))

    def forward(self, x):
        idt = self.short(x)
        out = self.drop(self.act(self.bn1(self.conv1(x))))
        out = self.bn2(self.conv2(out))
        return self.act(out + idt)


class XResNet1D(nn.Module):
    """Stem sâu (3 conv nhỏ xếp chồng) thay vì 1 conv to + SiLU — biến thể tối ưu hoá của ResNet1D."""

    def __init__(self, channels=(32, 64, 128, 256), dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(NUM_LEADS, 32, 5, 2, 2, bias=False), nn.BatchNorm1d(32), nn.SiLU(inplace=True),
            nn.Conv1d(32, 32, 5, 1, 2, bias=False), nn.BatchNorm1d(32), nn.SiLU(inplace=True),
            nn.Conv1d(32, 32, 5, 1, 2, bias=False), nn.BatchNorm1d(32), nn.SiLU(inplace=True),
            nn.MaxPool1d(2))
        blocks, c_in = [], 32
        for c_out in channels:
            blocks.append(XResBlock1D(c_in, c_out, stride=2))
            c_in = c_out
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Linear(c_in, 64), nn.SiLU(inplace=True),
                                  nn.Dropout(dropout), nn.Linear(64, 1))

    def forward(self, x):
        return self.head(self.blocks(self.stem(x))).squeeze(-1)


# ---------------------------------------------------------------- 9. ConvNeXtV2_1D
class GRN1D(nn.Module):
    """Global Response Normalization (ConvNeXt V2) — chuẩn hoá theo norm toàn cục của từng kênh."""

    def __init__(self, channels):
        super().__init__()
        self.gamma = nn.Parameter(torch.zeros(1, 1, channels))
        self.beta = nn.Parameter(torch.zeros(1, 1, channels))

    def forward(self, x):                          # x: (B, L, C)
        gx = torch.norm(x, p=2, dim=1, keepdim=True)
        nx = gx / (gx.mean(dim=-1, keepdim=True) + 1e-6)
        return self.gamma * (x * nx) + self.beta + x


class ConvNeXtV2Block1D(nn.Module):
    def __init__(self, channels, expand=4, dropout=0.1):
        super().__init__()
        self.dwconv = nn.Conv1d(channels, channels, 7, padding=3, groups=channels, bias=False)
        self.norm = nn.LayerNorm(channels)
        self.pw1 = nn.Linear(channels, channels * expand)
        self.act = nn.GELU()
        self.grn = GRN1D(channels * expand)
        self.pw2 = nn.Linear(channels * expand, channels)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):                           # x: (B, C, L)
        idt = x
        x = self.dwconv(x).transpose(1, 2)           # (B, L, C)
        x = self.norm(x)
        x = self.pw2(self.grn(self.act(self.pw1(x))))
        x = self.drop(x).transpose(1, 2)
        return x + idt


class ConvNeXtV2Downsample1D(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.norm = nn.BatchNorm1d(c_in)
        self.conv = nn.Conv1d(c_in, c_out, 2, stride=2, bias=False)

    def forward(self, x):
        return self.conv(self.norm(x))


class ConvNeXtV2_1D(nn.Module):
    """Khối depthwise-conv + LayerNorm + MLP mở rộng + GRN — bản 1D của ConvNeXt V2."""

    def __init__(self, channels=(32, 64, 128, 256), depths=(1, 1, 2, 1), dropout=0.1):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(NUM_LEADS, channels[0], 4, stride=4, bias=False),
                                  nn.BatchNorm1d(channels[0]))
        stages, c_in = [], channels[0]
        for stage_i, (c_out, depth) in enumerate(zip(channels, depths)):
            if stage_i > 0:
                stages.append(ConvNeXtV2Downsample1D(c_in, c_out))
            stages += [ConvNeXtV2Block1D(c_out, dropout=dropout) for _ in range(depth)]
            c_in = c_out
        self.stages = nn.Sequential(*stages)
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.LayerNorm(c_in), nn.Dropout(dropout), nn.Linear(c_in, 1))

    def forward(self, x):
        return self.head(self.stages(self.stem(x))).squeeze(-1)


# ---------------------------------------------------------------- 10. AiTiAMI (tái hiện)
class AiTiAMIBlock1D(nn.Module):
    """Residual block chuẩn, kernel hẹp hơn ResNet1D (5 thay vì 7) để bắt chi tiết QRS/ST mịn hơn."""

    def __init__(self, c_in, c_out, k=5, stride=2, dropout=0.15):
        super().__init__()
        self.conv1 = nn.Conv1d(c_in, c_out, k, stride, k // 2, bias=False)
        self.bn1 = nn.BatchNorm1d(c_out)
        self.conv2 = nn.Conv1d(c_out, c_out, k, 1, k // 2, bias=False)
        self.bn2 = nn.BatchNorm1d(c_out)
        self.drop = nn.Dropout(dropout)
        self.relu = nn.ReLU(inplace=True)
        self.short = (nn.Identity() if (stride == 1 and c_in == c_out)
                      else nn.Sequential(nn.Conv1d(c_in, c_out, 1, stride, bias=False),
                                         nn.BatchNorm1d(c_out)))

    def forward(self, x):
        idt = self.short(x)
        out = self.drop(self.relu(self.bn1(self.conv1(x))))
        return self.relu(self.bn2(self.conv2(out)) + idt)


class AiTiAMI(nn.Module):
    """Tái hiện theo MÔ TẢ KIẾN TRÚC trong Lee et al., Eur Heart J 2025 (ehaf004, ROMIAE):
    "an advanced algorithm... built on a residual neural network", đầu vào 500Hz 12 chuyển đạo.
    KHÔNG PHẢI model/trọng số gốc của Medical AI Co., Ltd — bài báo không công bố kiến trúc chi
    tiết (sản phẩm thương mại). Sâu hơn (5 stage) và rộng hơn ResNet1D, head avg+max-pool nối tiếp.
    """

    def __init__(self, channels=(48, 96, 192, 256, 320), dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(NUM_LEADS, 48, 15, 2, 7, bias=False),
                                  nn.BatchNorm1d(48), nn.ReLU(inplace=True), nn.MaxPool1d(2))
        blocks, c_in = [], 48
        for i, c_out in enumerate(channels):
            blocks.append(AiTiAMIBlock1D(c_in, c_out, stride=2 if i < 4 else 1))
            c_in = c_out
        self.blocks = nn.Sequential(*blocks)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.maxpool = nn.AdaptiveMaxPool1d(1)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(c_in * 2, 128), nn.ReLU(inplace=True),
                                  nn.Dropout(dropout), nn.Linear(128, 1))

    def forward(self, x):
        feat = self.blocks(self.stem(x))
        pooled = torch.cat([self.avgpool(feat), self.maxpool(feat)], dim=1)
        return self.head(pooled).squeeze(-1)




# ---------------------------------------------------------------- 11. S4D1D (state-space, diagonal)
class S4DKernel(nn.Module):
    """S4D theo Gu, Goel & Re 2022 (diagonal state-space) -- bản pure PyTorch dùng FFT, KHÔNG cần
    biên dịch kernel CUDA riêng như gói `s4`/`mamba-ssm` gốc (tránh rủi ro build lỗi trên Colab).
    Chậm hơn bản CUDA chính thức nhưng cùng công thức toán học."""

    def __init__(self, d_model, d_state=32, dt_min=1e-3, dt_max=1e-1):
        super().__init__()
        log_dt = torch.rand(d_model) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)
        self.log_dt = nn.Parameter(log_dt)
        self.log_A_real = nn.Parameter(torch.log(0.5 * torch.ones(d_model, d_state)))
        self.A_imag = nn.Parameter(math.pi * torch.arange(d_state).float().repeat(d_model, 1))
        self.C = nn.Parameter(torch.randn(d_model, d_state, 2) * 0.5 ** 0.5)

    def forward(self, L, device):
        dt = torch.exp(self.log_dt)                                     # (d_model,)
        A = -torch.exp(self.log_A_real) + 1j * self.A_imag               # (d_model, d_state)
        C = torch.view_as_complex(self.C)                                # (d_model, d_state)
        dtA = A * dt.unsqueeze(-1)                                       # (d_model, d_state)
        n = torch.arange(L, device=device)
        power = dtA.unsqueeze(-1) * n                                    # (d_model, d_state, L)
        C_eff = C * (torch.exp(dtA) - 1) / A
        K = 2 * torch.einsum('hn,hnl->hl', C_eff, torch.exp(power)).real  # (d_model, L)
        return K


class S4DBlock(nn.Module):
    def __init__(self, d_model, d_state=32, dropout=0.1):
        super().__init__()
        self.kernel = S4DKernel(d_model, d_state)
        self.D = nn.Parameter(torch.randn(d_model))
        self.norm = nn.LayerNorm(d_model)
        self.out_proj = nn.Sequential(nn.GELU(), nn.Linear(d_model, d_model), nn.Dropout(dropout))

    def forward(self, x):                          # x: (B, L, d_model)
        B, L, D = x.shape
        # cuFFT chi ho tro half-precision voi kich thuoc luy thua 2, nen ep FFT chay o float32
        # bat ke autocast dang bat (tranh RuntimeError khi L khong phai luy thua 2).
        with torch.autocast(device_type=x.device.type, enabled=False):
            K = self.kernel(L, x.device).float()        # (d_model, L)
            u = x.transpose(1, 2).float()                # (B, d_model, L)
            k_f = torch.fft.rfft(K, n=2 * L)
            u_f = torch.fft.rfft(u, n=2 * L)
            y = torch.fft.irfft(u_f * k_f, n=2 * L)[..., :L]
            y = y + u * self.D.unsqueeze(-1)
        y = y.to(x.dtype).transpose(1, 2)            # (B, L, d_model)
        return x + self.out_proj(self.norm(y))


class S4D1D(nn.Module):
    """CNN-stem hạ 5000 mẫu xuống ~312 rồi xếp 4 khối S4D. S4D lý thuyết xử lý được chuỗi dài
    trực tiếp không cần hạ mẫu, nhưng hạ mẫu trước giúp tiết kiệm bộ nhớ/thời gian đáng kể."""

    def __init__(self, d_model=128, n_layers=4, d_state=32, dropout=0.15):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(NUM_LEADS, d_model, 15, 4, 7, bias=False), nn.BatchNorm1d(d_model),
            nn.GELU(), nn.Conv1d(d_model, d_model, 7, 4, 3, bias=False), nn.BatchNorm1d(d_model),
            nn.GELU())                                                     # 5000 -> ~312
        self.layers = nn.ModuleList([S4DBlock(d_model, d_state, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Dropout(dropout), nn.Linear(d_model, 1))

    def forward(self, x):
        x = self.stem(x).transpose(1, 2)            # (B, L', d_model)
        for layer in self.layers:
            x = layer(x)
        return self.head(x.transpose(1, 2)).squeeze(-1)


# ---------------------------------------------------------------- 12. Mamba2Lite1D (selective SSM)
class SelectiveSSM1D(nn.Module):
    """Bản rút gọn thuần PyTorch của selective scan trong Mamba (Gu & Dao 2023) -- không dùng gói
    `mamba-ssm`/`causal-conv1d` (cần biên dịch CUDA, hay lỗi trên Colab). Chạy chậm hơn bản CUDA gốc
    (scan tuần tự theo thời gian bằng vòng lặp Python) nhưng cùng công thức: A cố định (âm),
    B/C/dt phụ thuộc input (tính "selective" đặc trưng của Mamba)."""

    def __init__(self, d_model, d_state=16, expand=2, dropout=0.1):
        super().__init__()
        d_inner = d_model * expand
        self.in_proj = nn.Linear(d_model, d_inner * 2)
        self.conv = nn.Conv1d(d_inner, d_inner, 3, padding=1, groups=d_inner)
        self.x_proj = nn.Linear(d_inner, d_state * 2 + 1)     # -> B, C, dt (theo từng bước thời gian)
        self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state + 1).float()).repeat(d_inner, 1))
        self.D = nn.Parameter(torch.ones(d_inner))
        self.out_proj = nn.Sequential(nn.Linear(d_inner, d_model), nn.Dropout(dropout))
        self.d_state = d_state

    def forward(self, x):                            # x: (B, L, d_model)
        Bsz, L, _ = x.shape
        xz = self.in_proj(x)
        u, z = xz.chunk(2, dim=-1)                    # (B, L, d_inner) mỗi cái
        u = self.conv(u.transpose(1, 2)).transpose(1, 2)[:, :L]
        u = nn.functional.silu(u)
        Bp, Cp, dt = torch.split(self.x_proj(u), [self.d_state, self.d_state, 1], dim=-1)
        dt = nn.functional.softplus(dt)                 # (B, L, 1), > 0
        A = -torch.exp(self.A_log)                      # (d_inner, d_state), < 0

        dA = torch.exp(dt.unsqueeze(-1) * A)             # (B, L, d_inner, d_state)
        dBu = dt.unsqueeze(-1) * Bp.unsqueeze(2) * u.unsqueeze(-1)   # (B, L, d_inner, d_state)

        h = torch.zeros(Bsz, u.shape[-1], self.d_state, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(L):                              # scan tuần tự theo thời gian
            h = dA[:, t] * h + dBu[:, t]
            ys.append(torch.einsum('bdn,bn->bd', h, Cp[:, t]))
        y = torch.stack(ys, dim=1) + u * self.D           # (B, L, d_inner)
        y = y * nn.functional.silu(z)
        return self.out_proj(y)


class MambaBlock1D(nn.Module):
    def __init__(self, d_model, d_state=16, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.ssm = SelectiveSSM1D(d_model, d_state, dropout=dropout)

    def forward(self, x):
        return x + self.ssm(self.norm(x))


class Mamba2Lite1D(nn.Module):
    """Xấp xỉ Mamba-2 bằng selective-SSM thuần PyTorch (KHÔNG PHẢI bản CUDA gốc của gói mamba-ssm,
    nên hiệu năng/tốc độ không đại diện đầy đủ cho Mamba-2 chính thức). CNN-stem hạ mẫu mạnh
    (5000 -> ~78) vì scan tuần tự theo thời gian tốn hơn convolution nhiều."""

    def __init__(self, d_model=96, n_layers=3, d_state=16, dropout=0.15):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(NUM_LEADS, d_model, 15, 4, 7, bias=False), nn.BatchNorm1d(d_model), nn.GELU(),
            nn.Conv1d(d_model, d_model, 7, 4, 3, bias=False), nn.BatchNorm1d(d_model), nn.GELU(),
            nn.Conv1d(d_model, d_model, 5, 4, 2, bias=False), nn.BatchNorm1d(d_model), nn.GELU())
        self.layers = nn.ModuleList([MambaBlock1D(d_model, d_state, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Dropout(dropout), nn.Linear(d_model, 1))

    def forward(self, x):
        x = self.stem(x).transpose(1, 2)
        for layer in self.layers:
            x = layer(x)
        return self.head(x.transpose(1, 2)).squeeze(-1)


# ---------------------------------------------------------------- 13. ECGMamba1D (CNN + Mamba hybrid)
class ECGMamba1D(nn.Module):
    """Xấp xỉ theo tinh thần ECGMamba (Liu et al. 2024): CNN rút đặc trưng cục bộ (hình dạng
    QRS/ST) rồi Mamba-lite mô hình hoá quan hệ dài hạn giữa các nhịp -- KHÔNG PHẢI mã nguồn gốc
    của paper (không có repo công khai xác minh được lúc viết notebook này), tự thiết kế lại
    theo đúng mô tả kiến trúc hai tầng CNN + Mamba."""

    def __init__(self, d_model=96, n_layers=2, d_state=16, dropout=0.15):
        super().__init__()
        layers, c_in = [], NUM_LEADS
        for c_out in (32, 64, d_model):
            layers += [nn.Conv1d(c_in, c_out, 7, padding=3, bias=False),
                       nn.BatchNorm1d(c_out), nn.GELU(), nn.MaxPool1d(4)]
            c_in = c_out
        self.cnn = nn.Sequential(*layers)              # 5000 -> ~78
        self.mamba_layers = nn.ModuleList(
            [MambaBlock1D(d_model, d_state, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Dropout(dropout), nn.Linear(d_model, 1))

    def forward(self, x):
        x = self.cnn(x).transpose(1, 2)
        for layer in self.mamba_layers:
            x = layer(x)
        return self.head(x.transpose(1, 2)).squeeze(-1)


# ---------------------------------------------------------------- 14. OMICNN1D (Queen-of-Hearts-style)
class OMICNN1D(nn.Module):
    """Lấy cảm hứng từ mô tả công khai của Queen of Hearts (Powerful Medical/PMcardio) -- một CNN
    sâu multi-scale phát hiện tắc mạch hoàn toàn (OMI). KHÔNG PHẢI bản sao kiến trúc/trọng số
    thương mại -- công ty không công bố mã nguồn hay chi tiết kiến trúc. Khác 8 model gốc ở chỗ:
    mỗi tầng có 3 nhánh kernel song song (5/11/21) để bắt cả biến dạng ST tinh vi lẫn hình thái
    QRS rộng, theo đúng cách tiếp cận multi-scale được mô tả trong các paper liên quan OMI/STEMI."""

    class MultiScaleBlock(nn.Module):
        def __init__(self, c_in, c_out, stride=2, dropout=0.15):
            super().__init__()
            k_each = c_out // 3
            self.branches = nn.ModuleList([
                nn.Conv1d(c_in, k_each, k, stride, k // 2, bias=False) for k in (5, 11, 21)])
            c_cat = k_each * 3
            self.bn = nn.BatchNorm1d(c_cat)
            self.act = nn.GELU()
            self.drop = nn.Dropout(dropout)
            self.proj = (nn.Identity() if c_cat == c_out else nn.Conv1d(c_cat, c_out, 1))
            self.short = (nn.Sequential(nn.Conv1d(c_in, c_out, 1, stride, bias=False),
                                        nn.BatchNorm1d(c_out)))

        def forward(self, x):
            feat = torch.cat([b(x) for b in self.branches], dim=1)
            feat = self.drop(self.act(self.bn(feat)))
            feat = self.proj(feat)
            return self.act(feat + self.short(x))

    def __init__(self, channels=(48, 96, 192, 256), dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(NUM_LEADS, 48, 15, 2, 7, bias=False),
                                  nn.BatchNorm1d(48), nn.GELU(), nn.MaxPool1d(2))
        blocks, c_in = [], 48
        for c_out in channels:
            blocks.append(self.MultiScaleBlock(c_in, c_out))
            c_in = c_out
        self.blocks = nn.Sequential(*blocks)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.maxpool = nn.AdaptiveMaxPool1d(1)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(c_in * 2, 128), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(128, 1))

    def forward(self, x):
        feat = self.blocks(self.stem(x))
        pooled = torch.cat([self.avgpool(feat), self.maxpool(feat)], dim=1)
        return self.head(pooled).squeeze(-1)


# ---------------------------------------------------------------- 15. OSCNN1D (Omni-Scale CNN)
def _os_cnn_prime_list(start, end):
    """Sàng nguyên tố trong [start, end] -- theo Tang et al. 2022, "Omni-Scale CNNs"."""
    primes = []
    for val in range(max(start, 2), end + 1):
        if all(val % n for n in range(2, val)):
            primes.append(val)
    return primes


def _os_cnn_layer_parameter_list(start, end, budget_per_layer, in_channel):
    """Sinh (in_ch, out_ch, kernel_size) cho từng layer: mỗi layer có 1 nhánh conv song song
    cho MỖI số nguyên tố trong [start, end] -- kernel size = số nguyên tố đó. out_ch của mỗi
    nhánh được suy từ ngân sách tham số `budget_per_layer` chia đều cho các nhánh, để độ rộng
    mạng không nổ khi số nguyên tố lớn. Không cần dò kernel size thủ công như CNN thường."""
    primes = _os_cnn_prime_list(start, end)
    if not primes:
        raise ValueError(f"Không có số nguyên tố nào trong [{start}, {end}]")
    layer_parameter_list = []
    for budget in budget_per_layer:
        out_channel = max(1, int(budget / (in_channel * sum(primes))))
        layer_parameter_list.append([(in_channel, out_channel, p) for p in primes])
        in_channel = len(primes) * out_channel
    # layer cuối: gộp bằng 2 nhánh kernel nhỏ (1 và 2) thay vì toàn bộ số nguyên tố
    first_out = len(primes) * max(1, int(budget_per_layer[0] / (in_channel * sum(primes))))
    layer_parameter_list.append([(in_channel, first_out, 1), (in_channel, first_out, 2)])
    return layer_parameter_list


class _SamePadConv1DBN(nn.Module):
    def __init__(self, c_in, c_out, kernel_size):
        super().__init__()
        self.pad = nn.ConstantPad1d(((kernel_size - 1) // 2, kernel_size // 2), 0.0)
        self.conv = nn.Conv1d(c_in, c_out, kernel_size)
        self.bn = nn.BatchNorm1d(c_out)

    def forward(self, x):
        return self.bn(self.conv(self.pad(x)))


class _OmniScaleLayer(nn.Module):
    """1 layer omni-scale: chạy song song 1 nhánh conv cho MỖI kernel size trong danh sách, rồi
    concat theo kênh + ReLU. Thay vì chọn 1 kernel size cố định (rủi ro bỏ sót scale phù hợp),
    omni-scale phủ toàn bộ dải kích thước nguyên tố trong [start, end] cùng lúc."""

    def __init__(self, layer_parameters):
        super().__init__()
        self.branches = nn.ModuleList(
            [_SamePadConv1DBN(c_in, c_out, k) for c_in, c_out, k in layer_parameters])

    def forward(self, x):
        return torch.relu(torch.cat([b(x) for b in self.branches], dim=1))


class OSCNN1D(nn.Module):
    """Omni-Scale CNN (Tang, Long, Liu, Yin 2022, arXiv:2002.10061) -- thay vì dò kernel size thủ
    công (như PlainCNN/ResNet1D), mỗi layer phủ TOÀN BỘ dải kích thước nguyên tố trong 1 khoảng
    tần số cho trước, được chứng minh đủ để phủ receptive field tối ưu cho time-series bất kỳ.
    Stem hạ mẫu 5000 -> ~625 trước (giữ chi phí tính toán hợp lý với ECG dài), sau đó 3 tầng
    omni-scale trên khoảng nguyên tố [1, ~156]."""

    def __init__(self, stem_channels=64, budget_per_layer=(8 * 64 * 32, 5 * 64 * 32, 5 * 64 * 16),
                 dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(NUM_LEADS, stem_channels, 15, 2, 7, bias=False), nn.BatchNorm1d(stem_channels),
            nn.GELU(), nn.Conv1d(stem_channels, stem_channels, 7, 4, 3, bias=False),
            nn.BatchNorm1d(stem_channels), nn.GELU())                      # 5000 -> ~625
        receptive_end = max(8, (SIGNAL_LEN // 8) // 4)                     # ~156
        layer_parameter_list = _os_cnn_layer_parameter_list(
            1, receptive_end, list(budget_per_layer), in_channel=stem_channels)
        self.layers = nn.Sequential(*[_OmniScaleLayer(p) for p in layer_parameter_list])
        out_channels = sum(c_out for _, c_out, _ in layer_parameter_list[-1])
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Dropout(dropout), nn.Linear(out_channels, 1))

    def forward(self, x):
        return self.head(self.layers(self.stem(x))).squeeze(-1)


MODELS = {
    "PlainCNN": PlainCNN,
    "ResNet1D": ResNet1D,
    "InceptionTime1D": InceptionTime1D,
    "CNN+BiLSTM": CNNBiLSTM,
    "SEResNet1D": SEResNet1D,
    "Transformer1D": Transformer1D,
    "TCN1D": TCN1D,
    "XResNet1D": XResNet1D,
    "ConvNeXtV2_1D": ConvNeXtV2_1D,
    "AiTiAMI": AiTiAMI,
    "S4D1D": S4D1D,
    "Mamba2Lite1D": Mamba2Lite1D,
    "ECGMamba1D": ECGMamba1D,
    "OMICNN1D": OMICNN1D,
    "OSCNN1D": OSCNN1D,
}


rows = []
for name, cls in MODELS.items():
    m = cls().to(DEVICE)
    with torch.no_grad():
        out = m(torch.zeros(2, NUM_LEADS, SIGNAL_LEN, device=DEVICE))
    rows.append({"Mô hình": name, "Tham số": sum(p.numel() for p in m.parameters()),
                 "Output": str(tuple(out.shape))})
    del m
torch.cuda.empty_cache() if GPU_AVAILABLE else None
display(pd.DataFrame(rows).set_index("Mô hình"))

## 12b. Trích tập TEST giữ riêng (15%, trước khi chia K-Fold)

Toàn bộ mục 13-21 (K-Fold, chọn model, chọn ngưỡng, hiệu chuẩn) từ giờ chỉ chạy trên **85% bệnh
nhân** (gọi là POOL) — 15% còn lại (TEST) bị **cách ly hoàn toàn**, không xuất hiện trong bất kỳ
bước train/chọn model/chọn ngưỡng/hiệu chuẩn nào. Đây là bước bắt buộc để có một tập đánh giá cổ
điển đúng nghĩa (train/val chọn mọi thứ, test chỉ đánh giá một lần cuối, không được nhìn thấy trước).

Chia theo bệnh nhân (như mọi chỗ khác trong notebook), seed riêng để không trùng với 5-fold hay lần
chia 85/15 gốc của notebook 03.

In [ ]:
TEST_SIZE = 0.15

pat_all = df.groupby(COL_PATIENT)[TARGET_LABEL].max().reset_index()
pat_pool, pat_test = train_test_split(pat_all, test_size=TEST_SIZE,
                                      stratify=pat_all[TARGET_LABEL], random_state=SEED + 200)
TEST_PATIENTS = set(pat_test[COL_PATIENT])
IS_TEST = df[COL_PATIENT].isin(TEST_PATIENTS).to_numpy()
POOL_IDX = df.index[~IS_TEST].to_numpy()
TEST_IDX = df.index[IS_TEST].to_numpy()

assert not (set(df.loc[POOL_IDX, COL_PATIENT]) & TEST_PATIENTS), "RÒ RỈ: bệnh nhân TEST lọt vào POOL"
assert len(POOL_IDX) + len(TEST_IDX) == len(df), "POOL + TEST không phủ đúng toàn bộ df"

display(pd.DataFrame({
    "tập": ["POOL (train+val, 5-fold)", "TEST (giữ riêng)"],
    "bệnh nhân": [len(pat_pool), len(pat_test)],
    "bản ghi": [len(POOL_IDX), len(TEST_IDX)],
    "dương": [int(y[POOL_IDX].sum()), int(y[TEST_IDX].sum())],
    "tỷ lệ dương": [f"{y[POOL_IDX].mean():.2%}", f"{y[TEST_IDX].mean():.2%}"],
}))
print("\nTừ đây, mục 13 (K-Fold) chỉ dùng POOL_IDX -- TEST_IDX không được chạm tới cho đến mục 21b.")

## 13. Chia K-Fold theo bệnh nhân

Thay lần chia 85/15 duy nhất bằng `StratifiedKFold` chạy trên **bảng cấp bệnh nhân** (nhãn của một
bệnh nhân = `max` nhãn các bản ghi của họ, đúng như mục 9). Chia ở cấp bệnh nhân rồi mới map ngược
về bản ghi là cách duy nhất đảm bảo cứng rằng không bệnh nhân nào xuất hiện ở hai fold — chia trực
tiếp theo bản ghi sẽ rò rỉ, vì 855 bệnh nhân có nhiều hơn một ECG.

**Cùng một bộ fold dùng cho cả ba ứng viên** — đó là điều kiện để so sánh ở mục 16 là *paired*
(so từng fold một), chứ không phải so hai giá trị trung bình rời rạc.

In [ ]:
from sklearn.model_selection import StratifiedKFold

pat_tbl = df.loc[POOL_IDX].groupby(COL_PATIENT)[TARGET_LABEL].max().reset_index()
POS_RATE_ALL = float(y[POOL_IDX].mean())

FOLDS = {}          # (rep, k) -> (train_idx, val_idx) -- CHỈ trong POOL_IDX, không đụng TEST_IDX
for rep in range(N_REPEATS):
    skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED + rep)
    for k, (_, va_pos) in enumerate(skf.split(pat_tbl, pat_tbl[TARGET_LABEL])):
        va_pats = set(pat_tbl.iloc[va_pos][COL_PATIENT])
        in_va_pool = df.loc[POOL_IDX, COL_PATIENT].isin(va_pats).to_numpy()
        FOLDS[(rep, k)] = (POOL_IDX[~in_va_pool], POOL_IDX[in_va_pool])

# --- AC1: không bệnh nhân nào ở hai fold, không ai lọt vào TEST, mọi bản ghi POOL được val đúng 1 lần ---
for rep in range(N_REPEATS):
    seen = set()
    for k in range(K_FOLDS):
        pats_k = set(df.loc[FOLDS[(rep, k)][1], COL_PATIENT])
        assert not (seen & pats_k), f"RÒ RỈ: bệnh nhân nằm ở hai fold (rep {rep}, fold {k})"
        assert not (pats_k & TEST_PATIENTS), f"RÒ RỈ: bệnh nhân TEST lọt vào fold (rep {rep}, fold {k})"
        seen |= pats_k
        assert not (set(df.loc[FOLDS[(rep, k)][0], COL_PATIENT]) & pats_k), \
            f"RÒ RỈ: bệnh nhân vừa ở train vừa ở val (rep {rep}, fold {k})"
    cover = np.concatenate([FOLDS[(rep, k)][1] for k in range(K_FOLDS)])
    assert len(cover) == len(POOL_IDX) and len(set(cover)) == len(POOL_IDX), \
        "Các fold không phủ đúng POOL_IDX một lần"
print(f"Giao nhau bệnh nhân giữa các fold: RỖNG -> OK ({N_REPEATS} lần lặp) | TEST không lọt vào fold nào -> OK")

# --- AC1: tỷ lệ dương mỗi fold lệch không quá 2 điểm % so với toàn tập ---
rows, worst = [], 0.0
for (rep, k), (tr, va) in FOLDS.items():
    rate = float(y[va].mean())
    worst = max(worst, abs(rate - POS_RATE_ALL))
    rows.append({"lặp": rep, "fold": k, "bệnh nhân val": df.loc[va, COL_PATIENT].nunique(),
                 "bản ghi train": len(tr), "bản ghi val": len(va),
                 "dương val": int(y[va].sum()), "tỷ lệ dương val": rate,
                 "lệch (điểm %)": (rate - POS_RATE_ALL) * 100})
fold_df = pd.DataFrame(rows)
display(fold_df.style.format({"tỷ lệ dương val": "{:.2%}", "lệch (điểm %)": "{:+.2f}"}))
assert worst <= 0.02, f"Fold lệch tỷ lệ dương tới {worst:.2%} > 2 điểm %"
print(f"Tỷ lệ dương toàn tập {POS_RATE_ALL:.2%} | lệch lớn nhất {worst * 100:.2f} điểm % -> OK")

## 14. Huấn luyện các ứng viên trên từng fold

Pipeline giữ **nguyên xi** notebook 03: `BCEWithLogitsLoss(pos_weight)`, AdamW, `ReduceLROnPlateau`
theo val AUPRC, early stopping, gradient clipping, AMP. Chỉ hai thứ tính lại theo từng fold vì bắt
buộc phải thế: `pos_weight` (phụ thuộc số ca dương trong train của fold đó) và `LEAD_MEAN/LEAD_STD`
(z-score fit trên train của fold đó).

Mỗi (model, fold) có checkpoint riêng kèm *fingerprint*. Run All lại notebook: cặp nào đã có
checkpoint khớp thì bỏ qua train, chỉ chạy cặp còn thiếu.

Dự đoán out-of-fold (OOF) được ghi ra `.npz`. Nếu file này đã đầy đủ, **toàn bộ mục 14 bị bỏ qua** —
các mục phân tích 15–19 chạy lại được trên máy không có GPU.

In [ ]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

OOF_DIR.mkdir(parents=True, exist_ok=True)
OOF_NPZ = OOF_DIR / f"oof_{TARGET_LABEL}_{RUN_MODE}_k{K_FOLDS}r{N_REPEATS}_holdout.npz"
OOF_JSON = OOF_NPZ.with_suffix(".meta.json")

CV_FP = dict(run_mode=RUN_MODE, target=TARGET_LABEL, k=K_FOLDS, reps=N_REPEATS,
             seed=SEED, epochs=EPOCHS, n=len(df), n_pool=len(POOL_IDX), models=CANDIDATES,
             n_folds_run=N_FOLDS_RUN)
# n_pool bắt buộc phải có trong fingerprint: checkpoint train TRƯỚC khi tách TEST (n_pool=len(df))
# sẽ không khớp CV_FP mới (n_pool=85% len(df)) -> tự động train lại, tránh rò rỉ TEST vào model.

MODEL_LR_OVERRIDE = {"Transformer1D": 3e-4}    # giữ nguyên từ notebook 03
GRAD_CLIP_NORM = 1.0

NAN = float("nan")
OOF_P = {m: np.full((N_REPEATS, len(df)), NAN) for m in CANDIDATES}
FOLD_ID = np.full((N_REPEATS, len(df)), -1, dtype=int)
RUN_INFO = {}


def evaluate(model, loader):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                logits = model(xb)
            ys.append(yb.numpy())
            ps.append(torch.sigmoid(logits.float()).cpu().numpy())
    y_out, p_out = np.concatenate(ys), np.concatenate(ps)
    bad = ~np.isfinite(p_out)
    if bad.any():
        print(f"  !! {bad.sum()} xác suất không hữu hạn -> thay bằng 0.5; model đang bất ổn định.")
        p_out = np.where(bad, 0.5, p_out)
    return y_out, p_out


def train_fold(name, cls, rep, k, tr_loader, va_loader, pos_weight, tag):
    """Bản sao train_one() của notebook 03, thêm tham số fold thay vì đọc biến toàn cục."""
    ckpt_path = MODEL_DIR / f"{name.replace('+', '_')}_r{rep}f{k}_best.pt"
    fp = dict(CV_FP, model=name, rep=rep, fold=k)
    fp.pop("models"), fp.pop("n_folds_run")

    if ckpt_path.exists():
        ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        if ck.get("fp") == fp:
            model = cls().to(DEVICE)
            try:
                model.load_state_dict(ck["model"])
            except RuntimeError as e:
                # fingerprint khớp nhưng shape lệch -> checkpoint từ một phiên bản kiến trúc CŨ
                # (vd sau khi sửa lại class model). Không thể tái dùng -> train lại từ đầu.
                print(f"  [{tag}] checkpoint có fp khớp nhưng kiến trúc lệch (đã sửa code model?) "
                      f"-> bỏ qua checkpoint, train lại. Chi tiết: {e}".splitlines()[0])
            else:
                print(f"  [{tag}] đã có checkpoint khớp -> bỏ qua train "
                      f"(best AUPRC {ck['best']:.4f} @ ep {ck['best_epoch']})")
                return model, ck["hist"], ck["best_epoch"], ck["seconds"]

    torch.manual_seed(SEED + rep)
    np.random.seed(SEED + rep)
    model = cls().to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=DEVICE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=MODEL_LR_OVERRIDE.get(name, LR),
                                  weight_decay=WEIGHT_DECAY)
    scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=.5, patience=LR_PATIENCE)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    best, best_epoch, bad, hist = -np.inf, -1, 0, []
    t_all = time.time()
    for epoch in range(EPOCHS):
        model.train()
        tot, seen = 0.0, 0
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                loss = criterion(model(xb), yb)
            if not torch.isfinite(loss):
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            tot += loss.item() * xb.size(0)
            seen += xb.size(0)

        va_y, va_p = evaluate(model, va_loader)
        m = compute_metrics(va_y, va_p)
        scheduler.step(m["auprc"] if not np.isnan(m["auprc"]) else -np.inf)
        hist.append({"epoch": epoch + 1, "train_loss": tot / max(seen, 1),
                     "val_auprc": m["auprc"], "val_auroc": m["auroc"]})

        if (not np.isnan(m["auprc"])) and m["auprc"] > best:
            best, best_epoch, bad = float(m["auprc"]), epoch + 1, 0
            torch.save({"model": model.state_dict(), "fp": fp, "best": best,
                        "best_epoch": best_epoch, "hist": hist,
                        "seconds": round(time.time() - t_all, 1)}, ckpt_path)
        else:
            bad += 1
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  [{tag}] ep {epoch + 1:>3}/{EPOCHS}  loss {tot / max(seen, 1):.4f}  "
                  f"val auprc {m['auprc']:.4f}  auroc {m['auroc']:.4f}")
        if bad >= EARLY_STOP_PATIENCE:
            print(f"  [{tag}] early stop @ epoch {epoch + 1}")
            break

    seconds = round(time.time() - t_all, 1)
    ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    ck["seconds"], ck["hist"] = seconds, hist
    torch.save(ck, ckpt_path)
    model.load_state_dict(ck["model"])
    print(f"  [{tag}] xong {len(hist)} epoch trong {seconds:.0f}s | best AUPRC {best:.4f} @ ep {best_epoch}")
    return model, hist, best_epoch, seconds


def _oof_ready():
    if not (OOF_NPZ.exists() and OOF_JSON.exists()):
        return False
    if json.loads(OOF_JSON.read_text()).get("fp") != CV_FP:
        return False
    z = np.load(OOF_NPZ, allow_pickle=False)
    return all(f"p_{m}" in z for m in CANDIDATES)


if _oof_ready():
    _z = np.load(OOF_NPZ, allow_pickle=False)
    for m in CANDIDATES:
        OOF_P[m] = _z[f"p_{m}"]
    FOLD_ID = _z["fold_id"]
    RUN_INFO = json.loads(OOF_JSON.read_text())["runs"]
    print(f"Đã có OOF đầy đủ: {OOF_NPZ.name} -> bỏ qua toàn bộ phần huấn luyện.")
else:
    for rep in range(N_REPEATS):
        for k in range(N_FOLDS_RUN):
            tr_idx_k, va_idx_k = FOLDS[(rep, k)]
            mean_k, std_k = norm_stats(tr_idx_k)
            tr_loader, va_loader = make_loaders(tr_idx_k, va_idx_k, mean_k, std_k)
            n_pos_k = int(y[tr_idx_k].sum())
            pw = float((len(tr_idx_k) - n_pos_k) / max(n_pos_k, 1))
            FOLD_ID[rep, va_idx_k] = k
            print(f"\n=== lặp {rep} · fold {k} " + "=" * 46 +
                  f"\n    train {len(tr_idx_k):,} ({n_pos_k} dương, pos_weight {pw:.2f}) | "
                  f"val {len(va_idx_k):,} ({int(y[va_idx_k].sum())} dương)")
            for name in CANDIDATES:
                tag = f"{name} r{rep}f{k}"
                model, hist, best_epoch, secs = train_fold(
                    name, MODELS[name], rep, k, tr_loader, va_loader, pw, tag)
                _, p = evaluate(model, va_loader)
                OOF_P[name][rep, va_idx_k] = p
                RUN_INFO[f"{name}|{rep}|{k}"] = {
                    "best_epoch": best_epoch, "seconds": secs, "n_epoch": len(hist),
                    "params": sum(q.numel() for q in model.parameters()),
                    "hist": hist}
                del model
                if GPU_AVAILABLE:
                    torch.cuda.empty_cache()

    np.savez_compressed(OOF_NPZ, y=y, fold_id=FOLD_ID,
                        patient=df[COL_PATIENT].astype(str).to_numpy(),
                        **{f"p_{m}": OOF_P[m] for m in CANDIDATES})
    OOF_JSON.write_text(json.dumps({"fp": CV_FP, "runs": RUN_INFO}, ensure_ascii=False))
    print(f"\nĐã ghi OOF -> {OOF_NPZ}")

VALID = np.isfinite(OOF_P[CANDIDATES[0]][0])     # bản ghi đã có dự đoán OOF (debug: chỉ fold 0)
print(f"OOF phủ {VALID.sum():,}/{len(df):,} bản ghi | {int(y[VALID].sum())} ca dương")

### 14b. Ensemble kiểu WBF trên các ứng viên

`WBF-ensemble` ở notebook 03 là tổ hợp trọng số của **cả mười/mười một** model, nên không thể
"huấn luyện lại" nó với chỉ vài model con. Ở đây dựng lại đúng công thức ấy nhưng trên các ứng viên —
trọng số tính **trong nội bộ từng fold** (theo AUROC của fold đó) để không mượn thông tin fold khác.
Đây là một điểm so sánh miễn phí, không phải mục tiêu tuning.

In [ ]:
_EPS = 1e-3
ENS_NAMES = ["WBF-CV", "Average-CV"]
for m in ENS_NAMES:
    OOF_P[m] = np.full((N_REPEATS, len(df)), NAN)

for rep in range(N_REPEATS):
    for k in range(N_FOLDS_RUN):
        m_fold = FOLD_ID[rep] == k
        if not m_fold.any():
            continue
        w_raw = {}
        for name in CANDIDATES:
            a = compute_metrics(y[m_fold], OOF_P[name][rep, m_fold])["auroc"]
            w_raw[name] = max((0.5 if np.isnan(a) else a) - 0.5, _EPS)
        w_sum = sum(w_raw.values())
        OOF_P["WBF-CV"][rep, m_fold] = sum(
            w_raw[n] / w_sum * OOF_P[n][rep, m_fold] for n in CANDIDATES)
        OOF_P["Average-CV"][rep, m_fold] = sum(
            OOF_P[n][rep, m_fold] for n in CANDIDATES) / len(CANDIDATES)

ALL_NAMES = CANDIDATES + ENS_NAMES
print("Đã dựng ensemble trong từng fold:", ", ".join(ENS_NAMES))

### 14c. Train lại một lần trên toàn bộ POOL (model "final", chuẩn cổ điển)

5-fold ở mục 14 dùng để **kiểm chứng cấu hình**, không tạo ra model để triển khai — mỗi checkpoint
chỉ thấy 80% POOL. Bước này train lại **đúng một lần mỗi kiến trúc, trên toàn bộ POOL (100%)** — dùng
số epoch cố định = **trung vị `best_epoch`** đã quan sát qua 5 fold (không còn tập val để tự dừng
sớm, vì đã kiểm chứng xong ở bước CV). Kết quả: đúng **5 checkpoint** (không phải 25), mỗi kiến trúc
một file `_FINAL_pool.pt` — đây mới là model đúng nghĩa "cuối cùng" theo quy trình cổ điển.

In [ ]:
mean_pool, std_pool = norm_stats(POOL_IDX)
n_pos_pool = int(y[POOL_IDX].sum())
pw_pool = float((len(POOL_IDX) - n_pos_pool) / max(n_pos_pool, 1))
pool_loader = DataLoader(ECGDataset(POOL_IDX, y[POOL_IDX], mean_pool, std_pool), batch_size=BATCH_SIZE,
                         shuffle=True, num_workers=NUM_WORKERS, pin_memory=GPU_AVAILABLE, **_extra)

FINAL_EPOCHS = {}
for name in CANDIDATES:
    eps = [RUN_INFO[f"{name}|0|{k}"]["best_epoch"] for k in range(N_FOLDS_RUN)
          if f"{name}|0|{k}" in RUN_INFO]
    FINAL_EPOCHS[name] = max(1, int(round(np.median(eps)))) if eps else EPOCHS
print("Số epoch dùng để train FINAL (trung vị best_epoch qua 5 fold):")
for name in CANDIDATES:
    print(f"  {name:<16} {FINAL_EPOCHS[name]} epoch")


def train_final(name, cls, n_epochs, tag):
    ckpt_path = MODEL_DIR / f"{name.replace('+', '_')}_FINAL_pool.pt"
    fp = dict(target=TARGET_LABEL, n_pool=len(POOL_IDX), model=name, n_epochs=n_epochs, seed=SEED)
    if ckpt_path.exists():
        ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        if ck.get("fp") == fp:
            model = cls().to(DEVICE)
            try:
                model.load_state_dict(ck["model"])
            except RuntimeError as e:
                print(f"  [{tag}] checkpoint FINAL có fp khớp nhưng kiến trúc lệch -> train lại. "
                      f"Chi tiết: {e}".splitlines()[0])
            else:
                print(f"  [{tag}] đã có checkpoint FINAL khớp -> bỏ qua train")
                return model

    torch.manual_seed(SEED + 500)
    model = cls().to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw_pool], device=DEVICE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=MODEL_LR_OVERRIDE.get(name, LR),
                                  weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    t0 = time.time()
    for epoch in range(n_epochs):
        model.train()
        for xb, yb in pool_loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                loss = criterion(model(xb), yb)
            if not torch.isfinite(loss):
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
        if (epoch + 1) % 10 == 0 or epoch == n_epochs - 1:
            print(f"  [{tag}] ep {epoch + 1:>3}/{n_epochs}  {time.time() - t0:.0f}s")
    torch.save({"model": model.state_dict(), "fp": fp}, ckpt_path)
    print(f"  [{tag}] xong {n_epochs} epoch trong {time.time() - t0:.0f}s -- đã lưu {ckpt_path.name}")
    return model


FINAL_MODELS = {}
for name in CANDIDATES:
    tag = f"{name} FINAL(pool,{FINAL_EPOCHS[name]}ep)"
    print(f"\n=== {tag} " + "=" * (50 - len(tag)))
    FINAL_MODELS[name] = train_final(name, MODELS[name], FINAL_EPOCHS[name], tag)
    if GPU_AVAILABLE:
        torch.cuda.empty_cache()

print(f"\nĐã có {len(FINAL_MODELS)} model FINAL (mỗi kiến trúc 1 checkpoint, train trên 100% POOL).")

## 15. Kết quả từng fold và độ ổn định

Ngưỡng của mỗi (model, fold) lấy theo Youden's J **tính riêng trong fold đó** — đúng cách notebook
03 làm, để bảng này so được với bảng cũ. Cột `Sensitivity`/`NPV` ở đây vì thế **không** dùng để
kết luận model nào recall tốt hơn; việc đó dành cho mục 17 (ép về cùng một Specificity).

In [ ]:
from sklearn.metrics import roc_curve


def youden_threshold(y_true, y_prob):
    if len(np.unique(y_true.astype(int))) < 2:
        return 0.5
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    return float(np.clip(thr[int(np.argmax(tpr - fpr))], 0, 1))


rows = []
for name in ALL_NAMES:
    for rep in range(N_REPEATS):
        for k in range(N_FOLDS_RUN):
            m_fold = FOLD_ID[rep] == k
            if not m_fold.any():
                continue
            yk, pk = y[m_fold], OOF_P[name][rep, m_fold]
            met = compute_metrics(yk, pk, youden_threshold(yk, pk))
            rows.append({"Model": name, "lặp": rep, "fold": k, "AUROC": met["auroc"],
                         "AUPRC": met["auprc"], "Sensitivity": met["sensitivity"],
                         "Specificity": met["specificity"], "F1": met["f1"], "NPV": met["npv"],
                         "Threshold": met["threshold"], "n": met["n"], "n_pos": met["n_pos"]})

FOLD_METRICS = pd.DataFrame(rows)
PCT_CV = ["AUROC", "AUPRC", "Sensitivity", "Specificity", "F1", "NPV", "Threshold"]
print("=" * 78)
print(f"KẾT QUẢ TỪNG FOLD — {CLASS_NAME} | {K_FOLDS} fold × {N_REPEATS} lặp")
print("=" * 78)
display(style_df(FOLD_METRICS.set_index(["Model", "lặp", "fold"]), PCT_CV))

agg = FOLD_METRICS.groupby("Model")[["AUROC", "AUPRC", "Sensitivity", "Specificity", "F1", "NPV"]]
summary_cv = agg.agg(["mean", "std"])
summary_cv.columns = [f"{c}_{s}" for c, s in summary_cv.columns]
summary_cv = summary_cv.sort_values("AUPRC_mean", ascending=False)
print("\nTrung bình ± độ lệch chuẩn qua các fold:")
display(style_df(summary_cv, [c for c in summary_cv.columns if c.endswith("_mean")]))

for name in summary_cv.index:
    print(f"  {name:<12} AUPRC {summary_cv.loc[name, 'AUPRC_mean']:.4f} "
          f"± {summary_cv.loc[name, 'AUPRC_std']:.4f}   "
          f"AUROC {summary_cv.loc[name, 'AUROC_mean']:.4f} "
          f"± {summary_cv.loc[name, 'AUROC_std']:.4f}")

CHAMPION = summary_cv.index[0]
CHALLENGER = summary_cv.index[1]
print(f"\n-> Hạng 1 theo AUPRC trung bình: {CHAMPION} | bám sát nhất: {CHALLENGER}")

# Chi phí huấn luyện
if RUN_INFO:
    cost = pd.DataFrame([{"Model": kk.split("|")[0], "Tham số": v["params"],
                          "Thời gian (s)": v["seconds"], "Số epoch": v["n_epoch"]}
                         for kk, v in RUN_INFO.items()])
    print("\nChi phí huấn luyện (tổng qua các fold):")
    display(cost.groupby("Model").agg({"Tham số": "first", "Thời gian (s)": "sum",
                                       "Số epoch": "mean"}).round(1))

## 16. Kiểm định: chênh lệch có thật hay là nhiễu?

**Một cảnh báo phải nói trước.** Yêu cầu ban đầu là "Wilcoxon signed-rank, kết luận p < 0,05". Với
**n = 5 cặp**, p **hai phía** nhỏ nhất mà kiểm định này có thể sinh ra là 2/2⁵ = **0,0625** — nghĩa
là dù model thắng cả 5/5 fold thì p vẫn không bao giờ xuống dưới 0,05. Ngưỡng đó bất khả thi về mặt
toán học ở K = 5, không phải vì model kém.

Vì giả thuyết vốn có hướng ("PlainCNN tốt hơn"), notebook báo cáo **Wilcoxon một phía** (p nhỏ nhất
0,03125 ở n = 5 — *có thể* < 0,05), kèm **paired t-test** và **bootstrap CI 95% của ΔAUPRC**. Cái
cuối mới là bằng chứng chính: nó không bị chặn bởi số fold, và nói thẳng khoảng chênh lệch thật sự
nằm ở đâu. Muốn Wilcoxon hai phía đủ lực thì đặt `N_REPEATS = 2` ở mục 2 (10 cặp, gấp đôi GPU).

In [ ]:
from scipy.stats import ttest_rel, wilcoxon

PAT_CODES = pd.factorize(df[COL_PATIENT].astype(str))[0]


def cluster_bootstrap_delta(name_a, name_b, n_boot=N_BOOTSTRAP, seed=SEED):
    """CI của ΔAUPRC, lấy mẫu lại THEO BỆNH NHÂN (một bệnh nhân có thể có nhiều bản ghi)."""
    mask = VALID
    idx_all = np.where(mask)[0]
    codes = PAT_CODES[idx_all]
    order = np.argsort(codes, kind="stable")
    idx_sorted, codes_sorted = idx_all[order], codes[order]
    starts = np.searchsorted(codes_sorted, np.unique(codes_sorted))
    groups = np.split(idx_sorted, starts[1:])
    rng = np.random.default_rng(seed)
    pa, pb = OOF_P[name_a][0], OOF_P[name_b][0]
    out = []
    for _ in range(n_boot):
        pick = rng.integers(0, len(groups), len(groups))
        sel = np.concatenate([groups[i] for i in pick])
        ys = y[sel]
        if len(np.unique(ys.astype(int))) < 2:
            continue
        out.append(average_precision_score(ys, pa[sel]) - average_precision_score(ys, pb[sel]))
    return np.asarray(out)


def compare_two(name_a, name_b):
    piv = FOLD_METRICS.pivot_table(index=["lặp", "fold"], columns="Model", values="AUPRC")
    a, b = piv[name_a].to_numpy(), piv[name_b].to_numpy()
    d = a - b
    n = len(d)
    print("=" * 78)
    print(f"{name_a}  vs  {name_b}   —   AUPRC trên {n} fold ghép cặp")
    print("=" * 78)
    display(pd.DataFrame({name_a: a, name_b: b, "Δ": d}, index=piv.index).round(4))
    print(f"{name_a} thắng {int((d > 0).sum())}/{n} fold | Δ trung bình {d.mean():+.4f} "
          f"(std {d.std(ddof=1):.4f})")

    if n < 2 or np.allclose(d, 0):
        print("Không đủ fold để kiểm định (chế độ debug?).")
        return
    p_one = wilcoxon(a, b, alternative="greater").pvalue
    p_two = wilcoxon(a, b, alternative="two-sided").pvalue
    p_t = ttest_rel(a, b).pvalue
    p_min_two = 2 / (2 ** n)
    print(f"\nWilcoxon một phía  p = {p_one:.4f}   (p nhỏ nhất có thể ở n={n}: {1 / 2 ** n:.5f})")
    print(f"Wilcoxon hai phía  p = {p_two:.4f}   (p nhỏ nhất có thể ở n={n}: {p_min_two:.5f}"
          f"{' — KHÔNG THỂ < 0,05' if p_min_two > 0.05 else ''})")
    print(f"Paired t-test      p = {p_t:.4f}")

    boot = cluster_bootstrap_delta(name_a, name_b)
    if len(boot):
        lo, hi = np.percentile(boot, [2.5, 97.5])
        print(f"Bootstrap ΔAUPRC (OOF gộp, lấy mẫu theo bệnh nhân, {len(boot)} lần): "
              f"{boot.mean():+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]")
    else:
        lo = hi = float("nan")

    print("\nKẾT LUẬN:")
    if p_one < 0.05 and lo > 0:
        print(f"  {name_a} tốt hơn {name_b} — khác biệt CÓ Ý NGHĨA (p một phía {p_one:.4f} < 0,05, "
              f"CI của Δ nằm trọn bên dương).")
    elif lo > 0:
        print(f"  Bootstrap CI của Δ nằm trọn bên dương nhưng Wilcoxon chưa đạt "
              f"(p {p_one:.4f}) — bằng chứng NGHIÊNG VỀ {name_a}, chưa chắc chắn ở mức 5%.")
    elif hi < 0:
        print(f"  {name_b} mới là bên tốt hơn — CI của Δ nằm trọn bên âm.")
    else:
        print(f"  KHÔNG ĐỦ BẰNG CHỨNG để phân biệt {name_a} và {name_b} — coi là NGANG NHAU. "
              f"CI của Δ chứa 0.")


for _other in [n for n in FOLD_METRICS["Model"].unique() if n != CHAMPION]:
    compare_two(CHAMPION, _other)
    print()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))
for j, metric in enumerate(["AUPRC", "AUROC"]):
    data = [FOLD_METRICS.loc[FOLD_METRICS["Model"] == n, metric].to_numpy() for n in ALL_NAMES]
    ax[j].boxplot(data, widths=.55, showmeans=True)   # không dùng labels=: bỏ ở matplotlib 3.9
    ax[j].set_xticks(range(1, len(ALL_NAMES) + 1))
    ax[j].set_xticklabels(ALL_NAMES)
    for i, vals in enumerate(data):
        ax[j].scatter(np.full(len(vals), i + 1) + np.random.uniform(-.08, .08, len(vals)),
                      vals, s=26, color="#E45756", zorder=3, alpha=.85)
    ax[j].set_title(f"{metric} theo fold — {CLASS_NAME}")
    ax[j].grid(alpha=.3, axis="y")
    ax[j].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()
print("Các hộp chồng lấn nhiều = chênh lệch trung bình không đáng tin; "
      "xem lại p-value và CI ở trên trước khi kết luận.")

## 17. So khớp ngưỡng: ép mọi model về cùng một điểm vận hành

`Sensitivity` ở bảng so sánh gốc được đo tại ngưỡng Youden's J **riêng của từng model**, nên hai model
có thể "recall khác nhau" chỉ vì ngưỡng của chúng khác nhau, không phải vì model khác nhau. Mục này
ép tất cả về **cùng Specificity = 0,85** (nội suy trên ROC) rồi mới so.

Hai bảng, **khác nguồn dữ liệu, không được đọc gộp**:

- **(a)** toàn bộ mười model của notebook 03 — nạp lại checkpoint cũ, chỉ chạy inference trên
  đúng tập val 85/15 gốc. Không huấn luyện gì thêm. Bảng này để *sàng* xem có ứng viên thứ tư không.
- **(b)** ba ứng viên + ensemble trên **OOF 5-fold** — chặt hơn hẳn, đây mới là bảng dùng để kết luận.

In [ ]:
def find_low_threshold(y_true, y_prob, target_npv=0.99):
    """Ngưỡng THẤP lớn nhất sao cho NPV (của nhóm dưới ngưỡng) vẫn >= target_npv."""
    grid = np.linspace(0.0, 0.5, 501)
    best = 0.0
    for t in grid:
        below = y_prob < t
        if below.sum() == 0:
            continue
        npv = 1 - y_true[below].mean()
        if npv >= target_npv:
            best = t
        else:
            break
    return best


def find_high_threshold(y_true, y_prob, min_group_frac=0.01):
    """Ngưỡng CAO nhỏ nhất sao cho nhóm 'cao' (>= ngưỡng) còn chiếm >= min_group_frac tập."""
    grid = np.linspace(0.99, 0.05, 191)
    n = len(y_true)
    for t in grid:
        above = y_prob >= t
        if above.sum() / n >= min_group_frac:
            ppv = float(y_true[above].mean()) if above.sum() else float("nan")
            return float(t), ppv
    return 0.5, float("nan")


def threshold_at_specificity(y_true, y_prob, target_spec):
    """Ngưỡng cho Specificity = target_spec, nội suy tuyến tính trên đường ROC."""
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    thr = np.clip(thr, 0, 1)                 # thr[0] là +inf theo quy ước của sklearn
    return float(np.interp(1 - target_spec, fpr, thr))


def matched_row(name, y_true, y_prob):
    t_spec = threshold_at_specificity(y_true, y_prob, SPEC_TARGET)
    m = compute_metrics(y_true, y_prob, t_spec)
    t_ro = find_low_threshold(y_true.astype(int), y_prob, RULE_OUT_NPV)
    below = y_prob < t_ro
    return {"Model": name, "AUROC": m["auroc"], "AUPRC": m["auprc"],
            f"Thr@Spec{SPEC_TARGET:.2f}": t_spec, "Sensitivity": m["sensitivity"],
            "Specificity": m["specificity"], "PPV": m["ppv"], "NPV": m["npv"],
            f"Thr rule-out (NPV≥{RULE_OUT_NPV:.0%})": t_ro,
            "% loại được": float(below.mean()),
            "ca dương bị bỏ sót": int(y_true[below].sum())}


MATCH_PCT = ["AUROC", "AUPRC", f"Thr@Spec{SPEC_TARGET:.2f}", "Sensitivity", "Specificity",
             "PPV", "NPV", f"Thr rule-out (NPV≥{RULE_OUT_NPV:.0%})", "% loại được"]

# ---------- (a) inference-only: checkpoint của notebook 03 trên val 85/15 gốc ----------
orig_rows, ORIG_P = [], {}
for _name, _cls in MODELS.items():
    _ck = ORIG_MODEL_DIR / f"{_name.replace('+', '_')}_best.pt"
    if not _ck.exists():
        continue
    try:
        _state = torch.load(_ck, map_location=DEVICE, weights_only=False)
        _m = _cls().to(DEVICE)
        _m.load_state_dict(_state["model"])
    except Exception as e:                    # kiến trúc lệch so với lúc train -> bỏ qua model đó
        print(f"  bỏ qua {_name}: {type(e).__name__} — {e}")
        continue
    _yv, _pv = evaluate(_m, val_loader)
    ORIG_P[_name] = _pv
    orig_rows.append(matched_row(_name, _yv, _pv))
    del _m
    if GPU_AVAILABLE:
        torch.cuda.empty_cache()

if orig_rows:
    _wbf_w = {}
    for _n, _p in ORIG_P.items():
        _a = compute_metrics(y[val_idx], _p)["auroc"]
        _wbf_w[_n] = max((0.5 if np.isnan(_a) else _a) - 0.5, 1e-3)
    _s = sum(_wbf_w.values())
    orig_rows.append(matched_row("WBF-ensemble", y[val_idx],
                                 sum(_wbf_w[n] / _s * p for n, p in ORIG_P.items())))
    orig_rows.append(matched_row("Average-ensemble", y[val_idx],
                                 sum(ORIG_P.values()) / len(ORIG_P)))
    orig_df = pd.DataFrame(orig_rows).set_index("Model").sort_values("Sensitivity", ascending=False)
    print("=" * 78)
    print(f"(a) SÀNG — {len(ORIG_P)} model của notebook 03, checkpoint cũ, val 85/15 gốc "
          f"({len(val_idx):,} bản ghi)")
    print("    Nguồn dữ liệu KHÁC bảng (b) — không so trực tiếp hai bảng với nhau.")
    print("=" * 78)
    display(style_df(orig_df, MATCH_PCT))

    _cand_best = orig_df.loc[[c for c in CANDIDATES if c in orig_df.index], "Sensitivity"].max()
    _extra_candidates = orig_df[(~orig_df.index.isin(CANDIDATES + ["WBF-ensemble", "Average-ensemble"]))
                                & (orig_df["Sensitivity"] > _cand_best + 0.03)]
    if len(_extra_candidates):
        print(f"\n!! ỨNG VIÊN THỨ TƯ: {', '.join(_extra_candidates.index)} vượt ứng viên tốt nhất "
              f"({_cand_best:.4f}) hơn 0,03 Sensitivity tại Spec={SPEC_TARGET} — vượt mức nhiễu "
              f"ước lượng. Đề nghị bổ sung vào vòng K-Fold (thêm vào CANDIDATES ở mục 2, chạy lại).")
    else:
        print(f"\nKhông model nào ngoài ba ứng viên vượt quá +0,03 Sensitivity tại "
              f"Spec={SPEC_TARGET} -> giữ nguyên danh sách ứng viên.")
else:
    print("!! Không tìm thấy checkpoint nào trong", ORIG_MODEL_DIR)
    print("   Bảng (a) bị bỏ qua. Muốn có bảng này thì copy checkpoint của notebook 03 vào đó")
    print("   (Colab: thư mục outputs/models trên Drive) rồi chạy lại ô này.")

# ---------- (b) OOF 5-fold, ba ứng viên + ensemble ----------
oof_rows = [matched_row(n, y[VALID], OOF_P[n][0][VALID]) for n in ALL_NAMES]
oof_df = pd.DataFrame(oof_rows).set_index("Model").sort_values("Sensitivity", ascending=False)
print("\n" + "=" * 78)
print(f"(b) KẾT LUẬN — OOF {K_FOLDS} fold, {int(VALID.sum()):,} bản ghi, mọi model tại "
      f"Specificity = {SPEC_TARGET}")
print("=" * 78)
display(style_df(oof_df, MATCH_PCT))

## 17b. Ép ngưỡng theo nhiều mức Sensitivity (91% – 95%)

Mục 17 ép mọi model về cùng **Specificity = 0,85** rồi đọc Sensitivity ra sao. Mục này làm **ngược
lại**: chọn trước một mức Sensitivity tối thiểu, xem Specificity/PPV/NPV còn lại bao nhiêu ở từng
mức — không train lại gì, chỉ chọn điểm khác trên đường ROC đã có sẵn từ `OOF_P`.

Thứ tự các dòng trong mỗi bảng **giữ nguyên theo AUPRC giảm dần** (AUPRC không phụ thuộc ngưỡng, nên
thứ tự này không đổi dù ép Sensitivity ở mức nào) — dễ so sánh xuyên suốt 5 bảng.

In [ ]:
def threshold_at_sensitivity(y_true, y_prob, target_sens):
    """Ngưỡng LỚN NHẤT sao cho Sensitivity thực tế vẫn >= target_sens — đảm bảo chặn dưới thật,
    không xấp xỉ bằng nội suy (nội suy tuyến tính trên đường ROC dạng bậc thang có thể cho kết quả
    tụt dưới mục tiêu, như đã thấy: 94,94% thay vì 95%). Dò trực tiếp trên điểm số của các ca dương,
    giống cách find_low_threshold/find_high_threshold đã làm."""
    y_true = np.asarray(y_true).astype(int)
    pos_scores = np.sort(y_prob[y_true == 1])[::-1]        # điểm ca dương, giảm dần
    n_pos = len(pos_scores)
    k = min(int(np.ceil(target_sens * n_pos)), n_pos)       # cần bắt ít nhất k ca dương
    return float(pos_scores[k - 1]) if k > 0 else 1.0


SENS_TARGETS = [0.91, 0.92, 0.93, 0.94, 0.95]
_DISPLAY_NAME = {"Average-CV": "Average Ensemble", "WBF-CV": "WBF Ensemble"}
_ROW_NAMES = ["Average-CV"] + list(CANDIDATES)    # Average Ensemble + 8 model đơn, không có WBF
SENS_COLS = ["AUROC", "AUPRC", "Sensitivity", "Specificity", "F1", "NPV", "PPV"]

# thứ tự cố định theo AUPRC giảm dần (không đổi theo ngưỡng) -> tính 1 lần, dùng lại cho cả 5 bảng
_auprc_order = sorted(
    _ROW_NAMES,
    key=lambda n: -compute_metrics(y[VALID], OOF_P[n][0][VALID])["auprc"])

for target in SENS_TARGETS:
    rows = []
    for name in _auprc_order:
        p_ = OOF_P[name][0][VALID]
        t = threshold_at_sensitivity(y[VALID], p_, target)
        m = compute_metrics(y[VALID], p_, t)
        rows.append({"Model": _DISPLAY_NAME.get(name, name), "AUROC": m["auroc"],
                     "AUPRC": m["auprc"], "Sensitivity": m["sensitivity"],
                     "Specificity": m["specificity"], "F1": m["f1"], "NPV": m["npv"],
                     "PPV": m["ppv"], "_thr": t})
    _sens_df = pd.DataFrame(rows).set_index("Model")

    print("=" * 78)
    print(f"ÉP NGƯỠNG Sensitivity >= {target:.0%}  —  OOF {K_FOLDS} fold, {int(VALID.sum()):,} bản ghi")
    print("=" * 78)
    display(style_df(_sens_df[SENS_COLS], SENS_COLS))
    _champ = _sens_df.index[0]
    _thr_v, _spec_v, _ppv_v = _sens_df.loc[_champ, "_thr"], _sens_df.loc[_champ, "Specificity"], _sens_df.loc[_champ, "PPV"]
    print(f"Ngưỡng {_champ}: {_thr_v:.4f}  |  Specificity còn lại {_spec_v:.4f}  |  PPV {_ppv_v:.4f}\n")

## 18. Hiệu chuẩn xác suất bằng cross-fit

Dự án dùng **xác suất liên tục** chứ không phải nhãn nhị phân cứng, nên xác suất phải đọc được như
xác suất thật: trong nhóm được chấm 0,30 thì đúng khoảng 30% là ca dương.

Yêu cầu gốc có một mâu thuẫn: bước 4 cấm hiệu chuẩn trên chính tập dùng để chọn ngưỡng ở bước 5,
nhưng bước 5 lại đòi bootstrap trên *toàn bộ* OOF. Cách gỡ ở đây là **cross-fit**: xác suất của fold
*i* được hiệu chuẩn bằng bộ hiệu chuẩn fit trên OOF của **4 fold còn lại**. Ghép lại thì mọi bản ghi
đều có xác suất "honest" — không bản ghi nào tham gia fit bộ hiệu chuẩn của chính nó — mà vẫn dùng
được hết dữ liệu cho mục 19, thay vì hy sinh một fold (~540 bản ghi, ~35 ca dương, quá ít cho isotonic).

In [ ]:
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression


def ece_table(y_true, y_prob, n_bins=ECE_BINS):
    """Copy nguyên từ mục 16b của notebook 03."""
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_prob = np.asarray(y_prob, dtype=float).ravel()
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(y_prob, edges[1:-1]), 0, n_bins - 1)
    ece, rows = 0.0, []
    for b in range(n_bins):
        m = idx == b
        if not m.sum():
            continue
        conf, acc = float(y_prob[m].mean()), float(y_true[m].mean())
        ece += m.sum() / len(y_prob) * abs(acc - conf)
        rows.append({"bin": f"[{edges[b]:.1f},{edges[b + 1]:.1f})", "n": int(m.sum()),
                     "xác suất TB": conf, "tỷ lệ dương thật": acc, "lệch": acc - conf})
    return float(ece), pd.DataFrame(rows)


def _logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))


def crossfit_calibrate(p_raw, method):
    """Fold i được hiệu chuẩn bằng bộ hiệu chuẩn fit trên OOF của các fold KHÁC."""
    out = np.full(len(p_raw), NAN)
    folds_present = [k for k in range(K_FOLDS) if (FOLD_ID[0] == k).any()]
    if len(folds_present) < 2:
        print("  (chỉ có 1 fold — không cross-fit được, fit và áp trên chính nó; chỉ dùng để test pipeline)")
        folds_present = folds_present or [0]
        m = FOLD_ID[0] == folds_present[0]
        fit_mask = apply_mask = m
        pairs = [(fit_mask, apply_mask)]
    else:
        pairs = [((FOLD_ID[0] != k) & VALID, FOLD_ID[0] == k) for k in folds_present]
    for fit_mask, apply_mask in pairs:
        if len(np.unique(y[fit_mask].astype(int))) < 2:
            out[apply_mask] = p_raw[apply_mask]
            continue
        if method == "platt":
            lr = LogisticRegression(C=1e6, solver="lbfgs", max_iter=1000)
            lr.fit(_logit(p_raw[fit_mask]).reshape(-1, 1), y[fit_mask].astype(int))
            out[apply_mask] = lr.predict_proba(_logit(p_raw[apply_mask]).reshape(-1, 1))[:, 1]
        else:
            iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
            iso.fit(p_raw[fit_mask], y[fit_mask].astype(int))
            out[apply_mask] = iso.predict(p_raw[apply_mask])
    return out


CAL_MODEL = CHAMPION
_p_raw = OOF_P[CAL_MODEL][0]
CAL_P = {"thô": _p_raw}
for _meth, _lbl in [("platt", "Platt"), ("isotonic", "Isotonic")]:
    print(f"Cross-fit {_lbl} cho {CAL_MODEL} ...")
    CAL_P[_lbl] = crossfit_calibrate(_p_raw, _meth)

ECE_ROWS, _tables = [], {}
for _lbl, _p in CAL_P.items():
    _e, _t = ece_table(y[VALID], _p[VALID])
    _tables[_lbl] = _t
    ECE_ROWS.append({"Hiệu chuẩn": _lbl, "ECE": _e,
                     "Brier": float(brier_score_loss(y[VALID].astype(int), _p[VALID])),
                     "AUROC": compute_metrics(y[VALID], _p[VALID])["auroc"],
                     "AUPRC": compute_metrics(y[VALID], _p[VALID])["auprc"]})
ece_df = pd.DataFrame(ECE_ROWS).set_index("Hiệu chuẩn")
print(f"\n{CAL_MODEL} — OOF {int(VALID.sum()):,} bản ghi")
display(style_df(ece_df, ["ECE", "Brier", "AUROC", "AUPRC"]))

_e0 = ece_df.loc["thô", "ECE"]
for _lbl in ["Platt", "Isotonic"]:
    _e1 = ece_df.loc[_lbl, "ECE"]
    print(f"  {_lbl:<9} ECE {_e0:.4f} -> {_e1:.4f}  "
          f"{'GIẢM (đạt AC)' if _e1 < _e0 else 'KHÔNG GIẢM (không đạt AC)'}")
BEST_CAL = min(["Platt", "Isotonic"], key=lambda l: ece_df.loc[l, "ECE"])
print(f"-> Dùng {BEST_CAL} cho phần phân tầng nguy cơ (ECE thấp nhất).")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
ax[0].plot([0, 1], [0, 1], "k--", lw=1, label="hiệu chuẩn hoàn hảo")
for _lbl, _c in zip(CAL_P, ["#E45756", "#4C78A8", "#54A24B"]):
    _t = _tables[_lbl]
    ax[0].plot(_t["xác suất TB"], _t["tỷ lệ dương thật"], marker="o", lw=2, color=_c,
               label=f"{_lbl} (ECE {ece_df.loc[_lbl, 'ECE']:.4f})")
ax[0].set_xlabel("xác suất mô hình dự đoán")
ax[0].set_ylabel("tỷ lệ dương tính thực tế")
ax[0].set_title(f"Reliability diagram — {CAL_MODEL} (OOF)")
ax[0].legend()
ax[0].grid(alpha=.3)
ax[1].hist([CAL_P["thô"][VALID], CAL_P[BEST_CAL][VALID]], bins=25, label=["thô", BEST_CAL],
           color=["#E45756", "#4C78A8"])
ax[1].set_title("Phân phối xác suất")
ax[1].set_xlabel("xác suất")
ax[1].legend()
ax[1].grid(alpha=.3)
plt.tight_layout()
plt.show()

print("\nBảng bin của bản đã hiệu chuẩn:")
display(_tables[BEST_CAL].round(4))

## 19. Phân tầng nguy cơ ba mức kèm khoảng tin cậy

Lặp lại đúng cách phân tầng ở mục 14c/15c của notebook 03 (ngưỡng thấp = mức cao nhất còn giữ
NPV ≥ 99%; ngưỡng cao = mức thấp nhất mà nhóm "cao" vẫn chiếm ≥ 1% tập), nhưng chạy trên xác suất
**đã hiệu chuẩn cross-fit** và kèm **95% CI** thay vì một con số điểm.

Bootstrap lấy mẫu lại **theo bệnh nhân**, không theo bản ghi: 855 bệnh nhân có nhiều hơn một ECG và
các bản ghi của cùng một người tương quan mạnh, nên lấy mẫu theo bản ghi sẽ coi chúng là độc lập và
cho ra khoảng tin cậy hẹp giả tạo. Ngưỡng giữ **cố định** ở giá trị chốt trên toàn bộ OOF — cái được
ước lượng khoảng là NPV/PPV *tại điểm vận hành đã chọn*, đúng như câu hỏi lâm sàng đặt ra.

In [ ]:
def patient_groups(mask):
    idx_all = np.where(mask)[0]
    codes = PAT_CODES[idx_all]
    order = np.argsort(codes, kind="stable")
    idx_sorted, codes_sorted = idx_all[order], codes[order]
    return np.split(idx_sorted, np.searchsorted(codes_sorted, np.unique(codes_sorted))[1:])


def tier_report(label, p_all):
    yv, pv = y[VALID].astype(int), p_all[VALID]
    t_low = find_low_threshold(yv, pv, RULE_OUT_NPV)
    t_high, _ = find_high_threshold(yv, pv)
    tier = np.where(pv < t_low, "Thấp", np.where(pv >= t_high, "Cao", "Trung bình"))
    rows = []
    for lvl in ["Thấp", "Trung bình", "Cao"]:
        m = tier == lvl
        rows.append({"Mức": lvl, "n": int(m.sum()), "tỷ lệ n": m.mean(),
                     "dương": int(yv[m].sum()), "tỷ lệ dương trong nhóm": _div(int(yv[m].sum()), int(m.sum()))})
    print("=" * 78)
    print(f"{label}  |  ngưỡng thấp {t_low:.3f}  |  ngưỡng cao {t_high:.3f}")
    print("=" * 78)
    display(pd.DataFrame(rows).set_index("Mức").style.format(
        {"tỷ lệ n": "{:.2%}", "tỷ lệ dương trong nhóm": "{:.2%}", "n": "{:,d}", "dương": "{:,d}"}))

    groups = patient_groups(VALID)
    pos_in = {i: j for j, i in enumerate(np.where(VALID)[0])}
    rng = np.random.default_rng(SEED)
    npv_b, ppv_b, frac_b = [], [], []
    for _ in range(N_BOOTSTRAP):
        pick = rng.integers(0, len(groups), len(groups))
        sel = np.concatenate([groups[i] for i in pick])
        loc = np.array([pos_in[i] for i in sel])
        ys, ps = yv[loc], pv[loc]
        low, high = ps < t_low, ps >= t_high
        npv_b.append(1 - ys[low].mean() if low.any() else NAN)
        ppv_b.append(ys[high].mean() if high.any() else NAN)
        frac_b.append(low.mean())
    ci = lambda a: np.nanpercentile(np.asarray(a, dtype=float), [2.5, 97.5])
    npv_pt = 1 - yv[pv < t_low].mean() if (pv < t_low).any() else NAN
    ppv_pt = yv[pv >= t_high].mean() if (pv >= t_high).any() else NAN
    n_lo, h_lo = ci(npv_b), ci(ppv_b)
    f_lo = ci(frac_b)
    print(f"\nBootstrap {N_BOOTSTRAP:,} lần, lấy mẫu lại theo {len(groups):,} bệnh nhân:")
    display(pd.DataFrame([
        {"Chỉ số": f"NPV nhóm THẤP (rule-out)", "Điểm": npv_pt,
         "95% CI": f"[{n_lo[0]:.4f}, {n_lo[1]:.4f}]"},
        {"Chỉ số": f"PPV nhóm CAO (rule-in)", "Điểm": ppv_pt,
         "95% CI": f"[{h_lo[0]:.4f}, {h_lo[1]:.4f}]"},
        {"Chỉ số": "% bệnh nhân loại được ở nhóm THẤP", "Điểm": float((pv < t_low).mean()),
         "95% CI": f"[{f_lo[0]:.4f}, {f_lo[1]:.4f}]"},
    ]).set_index("Chỉ số").style.format({"Điểm": "{:.4f}"}))
    return {"t_low": t_low, "t_high": t_high, "npv_ci": n_lo, "ppv_ci": h_lo}


TIER_RAW = tier_report(f"{CAL_MODEL} — xác suất THÔ (OOF)", CAL_P["thô"])
print()
TIER_CAL = tier_report(f"{CAL_MODEL} — sau hiệu chuẩn {BEST_CAL} (cross-fit)", CAL_P[BEST_CAL])

## 20. (Tuỳ chọn) Sweep hyperparameter nhẹ

Mặc định **tắt** (`RUN_SWEEP = False` ở mục 2) vì tốn thêm nhiều giờ GPU. Bật lên thì chạy hai giai đoạn:

1. **Thô** — learning rate ∈ {5e-4, 1e-3, 2e-3}, **chỉ fold 0**, cho từng ứng viên.
2. **Tinh** — với LR tốt nhất của từng model: dropout ∈ {0.2, 0.3, 0.4} × loss ∈ {pos_weight, focal γ=2},
   vẫn ở fold 0; tổ hợp thắng của mỗi model mới chạy đủ 5 fold.

Kết quả sweep **không** dùng lại được số liệu CV ở mục 14: những con số đó chạy ở hyperparameter mặc
định, không áp cho tổ hợp mới.

In [ ]:
import inspect
from functools import partial


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25):
        super().__init__()
        self.gamma, self.alpha = gamma, alpha

    def forward(self, logits, target):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, target, reduction="none")
        p_t = torch.exp(-bce)
        a_t = self.alpha * target + (1 - self.alpha) * (1 - target)
        return (a_t * (1 - p_t) ** self.gamma * bce).mean()


def train_variant(name, cls, rep, k, lr, loss_kind, pos_weight, tr_loader, va_loader, tag):
    """Như train_fold nhưng cho phép đổi LR / dropout / loss. Checkpoint riêng theo tag."""
    ckpt = MODEL_DIR / f"sweep_{tag.replace('+', '_').replace('=', '').replace(' ', '_')}.pt"
    fp = dict(CV_FP, model=name, rep=rep, fold=k, tag=tag)
    fp.pop("models"), fp.pop("n_folds_run")
    if ckpt.exists():
        ck = torch.load(ckpt, map_location=DEVICE, weights_only=False)
        if ck.get("fp") == fp:
            print(f"  [{tag}] có checkpoint khớp -> bỏ qua (AUPRC {ck['best']:.4f})")
            return ck["best"]

    torch.manual_seed(SEED + rep)
    model = cls().to(DEVICE)
    criterion = (FocalLoss() if loss_kind == "focal" else
                 nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=DEVICE)))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    sched = ReduceLROnPlateau(opt, mode="max", factor=.5, patience=LR_PATIENCE)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    best, bad = -np.inf, 0
    for epoch in range(EPOCHS):
        model.train()
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                loss = criterion(model(xb), yb)
            if not torch.isfinite(loss):
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(opt)
            scaler.update()
        m = compute_metrics(*evaluate(model, va_loader))
        sched.step(m["auprc"] if not np.isnan(m["auprc"]) else -np.inf)
        if (not np.isnan(m["auprc"])) and m["auprc"] > best:
            best, bad = float(m["auprc"]), 0
            torch.save({"model": model.state_dict(), "fp": fp, "best": best}, ckpt)
        else:
            bad += 1
            if bad >= EARLY_STOP_PATIENCE:
                break
    del model
    if GPU_AVAILABLE:
        torch.cuda.empty_cache()
    print(f"  [{tag}] best AUPRC {best:.4f}")
    return best


if not RUN_SWEEP:
    print("RUN_SWEEP = False -> bỏ qua sweep. Đặt True ở mục 2 để chạy (tốn nhiều giờ GPU).")
else:
    _tr0, _va0 = FOLDS[(0, 0)]
    _mean0, _std0 = norm_stats(_tr0)
    _trl, _val = make_loaders(_tr0, _va0, _mean0, _std0)
    _npos0 = int(y[_tr0].sum())
    _pw0 = float((len(_tr0) - _npos0) / max(_npos0, 1))

    sweep_rows, BEST_HP = [], {}
    for _name in CANDIDATES:
        _cls = MODELS[_name]
        # --- giai đoạn thô: learning rate, chỉ fold 0 ---
        _lr_scores = {}
        for _lr in [5e-4, 1e-3, 2e-3]:
            _tag = f"{_name} lr={_lr}"
            _lr_scores[_lr] = train_variant(_name, _cls, 0, 0, _lr, "bce", _pw0, _trl, _val, _tag)
            sweep_rows.append({"Model": _name, "Giai đoạn": "thô", "LR": _lr, "dropout": None,
                               "loss": "bce", "AUPRC fold0": _lr_scores[_lr]})
        _best_lr = max(_lr_scores, key=_lr_scores.get)

        # --- giai đoạn tinh: dropout × loss, vẫn fold 0 ---
        _has_dropout = "dropout" in inspect.signature(_cls.__init__).parameters
        _drops = [0.2, 0.3, 0.4] if _has_dropout else [None]
        if not _has_dropout:
            print(f"  ({_name} không nhận tham số dropout -> chỉ sweep loss)")
        _fine = {}
        for _dp in _drops:
            for _loss in ["bce", "focal"]:
                _mk = partial(_cls, dropout=_dp) if _dp is not None else _cls
                _tag = f"{_name} lr={_best_lr} dp={_dp} {_loss}"
                _sc = train_variant(_name, _mk, 0, 0, _best_lr, _loss, _pw0, _trl, _val, _tag)
                _fine[(_dp, _loss)] = _sc
                sweep_rows.append({"Model": _name, "Giai đoạn": "tinh", "LR": _best_lr,
                                   "dropout": _dp, "loss": _loss, "AUPRC fold0": _sc})
        _bd, _bl = max(_fine, key=_fine.get)
        BEST_HP[_name] = {"lr": _best_lr, "dropout": _bd, "loss": _bl}
        print(f"-> {_name}: lr={_best_lr}, dropout={_bd}, loss={_bl}")

    display(style_df(pd.DataFrame(sweep_rows).set_index(["Model", "Giai đoạn"]), ["AUPRC fold0"]))

    # --- tổ hợp thắng của từng model chạy đủ K fold ---
    full_rows = []
    for _name, _hp in BEST_HP.items():
        _cls = MODELS[_name]
        _mk = partial(_cls, dropout=_hp["dropout"]) if _hp["dropout"] is not None else _cls
        for _k in range(N_FOLDS_RUN):
            _tr, _va = FOLDS[(0, _k)]
            _m, _s = norm_stats(_tr)
            _tl, _vl = make_loaders(_tr, _va, _m, _s)
            _np_ = int(y[_tr].sum())
            _pw = float((len(_tr) - _np_) / max(_np_, 1))
            _tag = f"{_name} TUNED f{_k}"
            full_rows.append({"Model": f"{_name} (tuned)", "fold": _k,
                              "AUPRC": train_variant(_name, _mk, 0, _k, _hp["lr"], _hp["loss"],
                                                     _pw, _tl, _vl, _tag)})
    tuned_df = pd.DataFrame(full_rows)
    display(style_df(tuned_df.groupby("Model")["AUPRC"].agg(["mean", "std"]), ["mean"]))
    print("So với mục 15 (hyperparameter mặc định) để biết tuning có đáng hay không.")

## 21. Lưu ý khi kết luận

- **5 ứng viên, toàn bộ là kiến trúc MỚI chưa thử ở notebook 08** (`S4D1D`, `Mamba2Lite1D`,
  `ECGMamba1D`, `OMICNN1D`, `OSCNN1D`) — đây là những model duy nhất qua K-Fold trong notebook này.
  10 model của notebook 03 (gồm 8 model notebook 08 dùng, cộng `TCN1D`/`Transformer1D`) chỉ xuất
  hiện ở bảng sàng 17(a) với checkpoint cũ trên lần chia 85/15 gốc. Chúng **chưa** qua K-Fold ở đây,
  nên không có khoảng tin cậy và không được xếp hạng chung với 5 ứng viên mới.
- **Hai bảng ở mục 17 khác nguồn dữ liệu.** (a) là một lần chia 85/15; (b) là OOF 5 fold. Con số
  không so trực tiếp được với nhau.
- **Wilcoxon ở K = 5 không thể cho p hai phía < 0,05** (xem mục 16). Đọc kết luận dựa vào bootstrap
  CI của ΔAUPRC là chính.
- **Ngưỡng ở mục 19 chốt trên OOF, chưa phải trên dữ liệu độc lập.** Tập test ẩn 10% vẫn chưa được
  chạm tới ở bất kỳ bước nào của notebook này. Trước khi dùng lâm sàng, ngưỡng phải được xác nhận lại
  trên một tập độc lập.
- **Hiệu chuẩn phụ thuộc tỷ lệ mắc.** Tỷ lệ dương ở đây (8,05% STEMI) là của quần thể đã được chỉ định
  chụp mạch, cao hơn nhiều so với quần thể cấp cứu chung. Đưa sang nơi khác thì phải hiệu chuẩn lại.
- **Chưa làm:** kiểm tra ngoài phân phối trên Georgia / Chapman-Shaoxing (không có dữ liệu CinC2021
  trong `datasets/`) và phân tích lỗi false-negative kèm Grad-CAM — hai việc này để notebook riêng.

## 21b. Đánh giá cổ điển trên tập TEST giữ riêng (15%, chưa từng đụng tới)

Tập TEST (mục 12b) không tham gia bất kỳ bước nào ở trên — không train, không chọn ứng viên, không
chọn ngưỡng, không hiệu chuẩn. Đây là **lần duy nhất** nó được dùng, đúng tinh thần đánh giá cổ điển:
POOL (train/val, qua 5-fold) lo việc chọn model + ngưỡng + hiệu chuẩn; TEST chỉ đánh giá một lần cuối.

- **Xác suất trên TEST** cho mỗi kiến trúc = trung bình dự đoán của **cả 5 checkpoint fold** (bagging)
  — không train thêm model nào mới. Mỗi checkpoint chuẩn hoá dữ liệu bằng đúng z-score của fold nó
  từng train (không phải một bộ thống kê chung), khớp đúng cách nó được huấn luyện.
- **Ngưỡng và bộ hiệu chuẩn** dùng nguyên xi cái đã chốt từ OOF của POOL (mục 15-19) — **không refit
  lại trên TEST**, để tránh rò rỉ thông tin TEST vào bước chọn ngưỡng.

In [ ]:
# ---- 1) Bagging: mỗi kiến trúc = trung bình 5 checkpoint fold, dự đoán trên TEST ----
_fold_stats = {k: norm_stats(FOLDS[(0, k)][0]) for k in range(N_FOLDS_RUN)}
TEST_P = {}
for name in CANDIDATES:
    preds = []
    for k in range(N_FOLDS_RUN):
        mean_k, std_k = _fold_stats[k]
        ckpt_path = MODEL_DIR / f"{name.replace('+', '_')}_r0f{k}_best.pt"
        if not ckpt_path.exists():
            print(f"  !! thiếu checkpoint {ckpt_path.name} -- bỏ qua fold {k} của {name}")
            continue
        ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        model = MODELS[name]().to(DEVICE)
        model.load_state_dict(ck["model"])
        test_ds = ECGDataset(TEST_IDX, y[TEST_IDX], mean_k, std_k)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                                 num_workers=NUM_WORKERS, pin_memory=GPU_AVAILABLE, **_extra)
        _, p = evaluate(model, test_loader)
        preds.append(p)
        del model
        if GPU_AVAILABLE:
            torch.cuda.empty_cache()
    TEST_P[name] = np.mean(preds, axis=0)
    print(f"  {name}: bagging {len(preds)}/{N_FOLDS_RUN} fold checkpoint xong")

y_test = y[TEST_IDX].astype(int)
TEST_P["Average-CV"] = np.mean([TEST_P[n] for n in CANDIDATES], axis=0)
_pool_w_raw = {}
for name in CANDIDATES:
    _a = compute_metrics(y[VALID], OOF_P[name][0][VALID])["auroc"]
    _pool_w_raw[name] = max((0.5 if np.isnan(_a) else _a) - 0.5, _EPS)
_pool_w_sum = sum(_pool_w_raw.values())
TEST_P["WBF-CV"] = sum(_pool_w_raw[n] / _pool_w_sum * TEST_P[n] for n in CANDIDATES)
print(f"\nTEST: {len(TEST_IDX):,} bản ghi, {int(y_test.sum())} dương ({y_test.mean():.2%})")

# ---- 1b) Model FINAL (train 1 lần trên 100% POOL, mục 14c) -- dự đoán trên TEST ----
TEST_P_FINAL = {}
_test_ds_final = ECGDataset(TEST_IDX, y[TEST_IDX], mean_pool, std_pool)
_test_loader_final = DataLoader(_test_ds_final, batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=GPU_AVAILABLE, **_extra)
for name in CANDIDATES:
    _, p = evaluate(FINAL_MODELS[name], _test_loader_final)
    TEST_P_FINAL[name] = p
TEST_P_FINAL["Average-CV"] = np.mean([TEST_P_FINAL[n] for n in CANDIDATES], axis=0)
TEST_P_FINAL["WBF-CV"] = sum(_pool_w_raw[n] / _pool_w_sum * TEST_P_FINAL[n] for n in CANDIDATES)

print("\n" + "=" * 78)
print("SO SÁNH TRÊN TEST: BAGGING (5 checkpoint/kiến trúc) vs FINAL (1 checkpoint/kiến trúc)")
print("=" * 78)
_cmp_rows = []
for name in CANDIDATES + ["Average-CV", "WBF-CV"]:
    _m_bag = compute_metrics(y_test, TEST_P[name])
    _m_fin = compute_metrics(y_test, TEST_P_FINAL[name])
    _cmp_rows.append({"Model": name, "AUROC bagging": _m_bag["auroc"], "AUROC final": _m_fin["auroc"],
                      "AUPRC bagging": _m_bag["auprc"], "AUPRC final": _m_fin["auprc"]})
display(style_df(pd.DataFrame(_cmp_rows).set_index("Model"),
                 ["AUROC bagging", "AUROC final", "AUPRC bagging", "AUPRC final"]))
print("\nTừ đây, hai bảng metric/hiệu chuẩn/phân tầng dưới đây dùng BAGGING (TEST_P) làm chính --")
print("đổi sang TEST_P_FINAL nếu muốn báo cáo theo model FINAL thay vì bagging.")

# ---- 2) Ngưỡng & bộ hiệu chuẩn: chốt từ POOL OOF, áp nguyên xi lên TEST (không refit) ----
_y_pool, _p_pool = y[VALID].astype(int), OOF_P[CHAMPION][0][VALID]
t_youden_pool = youden_threshold(_y_pool, _p_pool)
t_spec_pool = threshold_at_specificity(_y_pool, _p_pool, SPEC_TARGET)


def fit_calibrator(y_fit, p_fit, method):
    if method == "Platt":
        lr = LogisticRegression(C=1e6, solver="lbfgs", max_iter=1000)
        lr.fit(_logit(p_fit).reshape(-1, 1), y_fit)
        return lambda p: lr.predict_proba(_logit(p).reshape(-1, 1))[:, 1]
    iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
    iso.fit(p_fit, y_fit)
    return lambda p: iso.predict(p)


_final_cal = fit_calibrator(_y_pool, _p_pool, BEST_CAL)
TEST_P_CAL = _final_cal(TEST_P[CHAMPION])

# ---- 3) Bảng metric cổ điển tại 2 điểm neo (ngưỡng chốt từ POOL), cho mọi ứng viên ----
def test_row(name, threshold):
    m = compute_metrics(y_test, TEST_P[name], threshold)
    return {"Model": name, "AUROC": m["auroc"], "AUPRC": m["auprc"], "Sensitivity": m["sensitivity"],
            "Specificity": m["specificity"], "PPV": m["ppv"], "NPV": m["npv"], "F1": m["f1"]}


TEST_COLS = ["AUROC", "AUPRC", "Sensitivity", "Specificity", "PPV", "NPV", "F1"]
_all_test_names = CANDIDATES + ["Average-CV", "WBF-CV"]

print("\n" + "=" * 78)
print(f"KẾT QUẢ TEST -- ngưỡng Youden's J của {CHAMPION}, chốt từ POOL ({t_youden_pool:.4f})")
print("=" * 78)
display(style_df(pd.DataFrame([test_row(n, t_youden_pool) for n in _all_test_names])
                 .set_index("Model"), TEST_COLS))

print("\n" + "=" * 78)
print(f"KẾT QUẢ TEST -- ngưỡng Specificity={SPEC_TARGET} của {CHAMPION}, chốt từ POOL ({t_spec_pool:.4f})")
print("=" * 78)
display(style_df(pd.DataFrame([test_row(n, t_spec_pool) for n in _all_test_names])
                 .set_index("Model"), TEST_COLS))

# ---- 4) Hiệu chuẩn trên TEST ----
_ece_raw, _ = ece_table(y_test, TEST_P[CHAMPION])
_ece_cal, _ = ece_table(y_test, TEST_P_CAL)
_ece_verdict = "GIẢM" if _ece_cal < _ece_raw else "KHÔNG GIẢM"
print(f"\nHiệu chuẩn {CHAMPION} trên TEST -- ECE thô {_ece_raw:.4f} -> sau {BEST_CAL} "
      f"{_ece_cal:.4f}  ({_ece_verdict})")

# ---- 5) Ngưỡng rule-out/rule-in chốt từ POOL (mục 19), đo lại trên TEST kèm bootstrap CI ----
def test_tier_report(label, p_all, t_low, t_high):
    tier = np.where(p_all < t_low, "Thấp", np.where(p_all >= t_high, "Cao", "Trung bình"))
    rows = []
    for lvl in ["Thấp", "Trung bình", "Cao"]:
        m = tier == lvl
        rows.append({"Mức": lvl, "n": int(m.sum()), "tỷ lệ n": m.mean(),
                     "dương": int(y_test[m].sum()),
                     "tỷ lệ dương trong nhóm": _div(int(y_test[m].sum()), int(m.sum()))})
    print("=" * 78)
    print(f"{label}  |  ngưỡng thấp {t_low:.3f} (POOL)  |  ngưỡng cao {t_high:.3f} (POOL)")
    print("=" * 78)
    display(pd.DataFrame(rows).set_index("Mức").style.format(
        {"tỷ lệ n": "{:.2%}", "tỷ lệ dương trong nhóm": "{:.2%}", "n": "{:,d}", "dương": "{:,d}"}))

    test_mask_full = np.zeros(len(df), dtype=bool)
    test_mask_full[TEST_IDX] = True
    groups = patient_groups(test_mask_full)
    pos_in = {i: j for j, i in enumerate(TEST_IDX)}
    rng = np.random.default_rng(SEED + 999)
    npv_b, ppv_b = [], []
    for _ in range(N_BOOTSTRAP):
        pick = rng.integers(0, len(groups), len(groups))
        sel = np.concatenate([groups[i] for i in pick])
        loc = np.array([pos_in[i] for i in sel])
        ys, ps = y_test[loc], p_all[loc]
        low, high = ps < t_low, ps >= t_high
        npv_b.append(1 - ys[low].mean() if low.any() else NAN)
        ppv_b.append(ys[high].mean() if high.any() else NAN)
    ci = lambda a: np.nanpercentile(np.asarray(a, dtype=float), [2.5, 97.5])
    npv_pt = 1 - y_test[p_all < t_low].mean() if (p_all < t_low).any() else NAN
    ppv_pt = y_test[p_all >= t_high].mean() if (p_all >= t_high).any() else NAN
    n_lo, h_lo = ci(npv_b), ci(ppv_b)
    print(f"\nBootstrap {N_BOOTSTRAP:,} lần trên TEST, lấy mẫu theo {len(groups):,} bệnh nhân:")
    display(pd.DataFrame([
        {"Chỉ số": "NPV nhóm THẤP (rule-out)", "Điểm": npv_pt,
         "95% CI": f"[{n_lo[0]:.4f}, {n_lo[1]:.4f}]"},
        {"Chỉ số": "PPV nhóm CAO (rule-in)", "Điểm": ppv_pt,
         "95% CI": f"[{h_lo[0]:.4f}, {h_lo[1]:.4f}]"},
    ]).set_index("Chỉ số").style.format({"Điểm": "{:.4f}"}))


test_tier_report(f"{CHAMPION} -- TEST, xác suất THÔ", TEST_P[CHAMPION],
                 TIER_RAW["t_low"], TIER_RAW["t_high"])
print()
test_tier_report(f"{CHAMPION} -- TEST, sau hiệu chuẩn {BEST_CAL}", TEST_P_CAL,
                 TIER_CAL["t_low"], TIER_CAL["t_high"])

In [ ]:
# ---- mục 21c: Ép ngưỡng Sensitivity 91-95% trên TEST giữ riêng (kết quả cuối cùng) ----
_test_auprc_order = sorted(
    _ROW_NAMES,
    key=lambda n: -compute_metrics(y_test, TEST_P[n])["auprc"])

for target in SENS_TARGETS:
    rows = []
    for name in _test_auprc_order:
        p_pool_ = OOF_P[name][0][VALID]
        t = threshold_at_sensitivity(y[VALID], p_pool_, target)   # chốt ngưỡng từ POOL
        m = compute_metrics(y_test, TEST_P[name], t)              # áp lên TEST, không refit

        rows.append({"Model": _DISPLAY_NAME.get(name, name), "AUROC": m["auroc"],
                     "AUPRC": m["auprc"], "Sensitivity": m["sensitivity"],
                     "Specificity": m["specificity"], "F1": m["f1"], "NPV": m["npv"],
                     "PPV": m["ppv"], "_thr": t})
    _sens_test_df = pd.DataFrame(rows).set_index("Model")

    print("=" * 78)
    print(f"ÉP NGƯỠNG Sensitivity >= {target:.0%}  —  TEST giữ riêng, {len(TEST_IDX):,} bản ghi")
    print("=" * 78)
    display(style_df(_sens_test_df[SENS_COLS], SENS_COLS))
    _champ = _sens_test_df.index[0]
    _thr_v, _spec_v, _ppv_v = (_sens_test_df.loc[_champ, "_thr"],
                               _sens_test_df.loc[_champ, "Specificity"],
                               _sens_test_df.loc[_champ, "PPV"])
    print(f"Ngưỡng {_champ}: {_thr_v:.4f}  |  Specificity còn lại {_spec_v:.4f}  |  PPV {_ppv_v:.4f}\n")


In [ ]:
# ---- mục 21d: Ép ngưỡng Sensitivity 91-95% trên TEST -- dùng model FINAL (1 checkpoint, không bagging) ----
_test_final_auprc_order = sorted(
    _ROW_NAMES,
    key=lambda n: -compute_metrics(y_test, TEST_P_FINAL[n])["auprc"])

for target in SENS_TARGETS:
    rows = []
    for name in _test_final_auprc_order:
        p_pool_ = OOF_P[name][0][VALID]
        t = threshold_at_sensitivity(y[VALID], p_pool_, target)   # chốt ngưỡng từ POOL (giống mục 21c)
        m = compute_metrics(y_test, TEST_P_FINAL[name], t)        # áp lên TEST, dùng model FINAL

        rows.append({"Model": _DISPLAY_NAME.get(name, name), "AUROC": m["auroc"],
                     "AUPRC": m["auprc"], "Sensitivity": m["sensitivity"],
                     "Specificity": m["specificity"], "F1": m["f1"], "NPV": m["npv"],
                     "PPV": m["ppv"], "_thr": t})
    _sens_test_final_df = pd.DataFrame(rows).set_index("Model")

    print("=" * 78)
    print(f"ÉP NGƯỠNG Sensitivity >= {target:.0%} -- MODEL FINAL  —  TEST giữ riêng, {len(TEST_IDX):,} bản ghi")
    print("=" * 78)
    display(style_df(_sens_test_final_df[SENS_COLS], SENS_COLS))
    _champ = _sens_test_final_df.index[0]
    _thr_v, _spec_v, _ppv_v = (_sens_test_final_df.loc[_champ, "_thr"],
                               _sens_test_final_df.loc[_champ, "Specificity"],
                               _sens_test_final_df.loc[_champ, "PPV"])
    print(f"Ngưỡng {_champ}: {_thr_v:.4f}  |  Specificity còn lại {_spec_v:.4f}  |  PPV {_ppv_v:.4f}\n")
